# SRF Qubit + Cavity Calibration

Calibration notebook for a fixed-frequency transmon coupled to two SRF cavities (Alice and Bob) on OPX+/Octave hardware.

**Calibration flow**:
1. Create / populate SRF QUAM state
2. Device setup (mixer, TOF)
3. Readout resonator
4. Transmon ge calibration
5. Transmon ef calibration
6. Dispersive shift (chi)
7. Alice cavity: spectroscopy → Rabi → T1
8. Bob cavity: spectroscopy → Rabi → T1
9. Automated calibration graphs

> **Important**: Run the preamble cell (Section 0) first in every session.

## 0. Preamble: run this first every session

In [1]:
from qualibrate_config.resolvers import get_qualibrate_config, get_qualibrate_config_path
from qualibrate_config.core.project.switch import switch_project

config_path = get_qualibrate_config_path()
config = get_qualibrate_config(config_path)
print(f"Current project: {config.project}")

desired_project = "calib_1q"
if config.project != desired_project:
    switch_project(config_path, desired_project)
    config = get_qualibrate_config(config_path)
    print(f"Switched to project: {config.project}")
else:
    print(f"Project already set to '{desired_project}'")

print(f"Storage location: {config.storage.location}")

Current project: calib_1q
Project already set to 'calib_1q'
Storage location: D:\MData\DR3-Run009\qm_calib\storage


In [2]:
%matplotlib widget

import sys
from quam_config import Quam, TemporaryCalibrationData


2026-04-12 12:52:48,630 - qm - INFO     - Starting session: 7ef8f4ee-ac48-4d3c-b797-40d1d450a784


## 1. Create QUAM state

Run **once** to build `state.json` and `wiring.json` in `quam_state/`.  Skip if the files already exist.

In [ ]:
import matplotlib.pyplot as plt
from qualang_tools.wirer.wirer.channel_specs import octave_spec, opx_iq_octave_spec, opx_dig_spec, ChannelSpecOctaveDigital
from qualang_tools.wirer import Instruments, Connectivity, allocate_wiring, visualize
from quam_builder.builder.qop_connectivity import build_quam_wiring
from quam_builder.builder.superconducting import build_quam
from quam.components.channels import DigitalOutputChannel
from quam_config import Quam

# ## Static parameters ####################################################
host_ip      = "192.168.3.50"   # OPX host IP
port         = None
cluster_name = "Cluster_1"
calibration_db_path = None

# ## Instruments ##########################################################
instruments = Instruments()
instruments.add_opx_plus(controllers=[1])
instruments.add_octave(indices=1)

qubits = [1]

# ## Channel addresses ####################################################
# Hardware routing:
#   Resonator  → RF_outputs/1 (int LO synth1), OPX+ ports 1(I)/2(Q), trigger 1
#   f0g1       → RF_outputs/2 (ext LO 3 GHz),  OPX+ ports 3(I)/4(Q), trigger 3  ← added in populate cell
#   Qubit XY   → RF_outputs/3 (int LO synth3),  OPX+ ports 5(I)/6(Q), trigger 5
#   Cavity     → RF_outputs/4 (int LO synth4),  OPX+ ports 7(I)/8(Q), trigger 7  ← added in populate cell
qubit_res_ch = opx_iq_octave_spec(con=1,
                               out_port_i=1, out_port_q=2,
                               in_port_i=1, in_port_q=2,
                               octave_index=1, rf_out=1, rf_in=1) &\
            opx_dig_spec(con=1, out_port=1) & ChannelSpecOctaveDigital(con=1, in_port=1)
qubit_xy_ch = opx_iq_octave_spec(con=1,
                             out_port_i=5, out_port_q=6,
                             octave_index=1, rf_out=3) &\
          opx_dig_spec(con=1, out_port=5) & ChannelSpecOctaveDigital(con=1, in_port=3)

# ## Allocate wiring ######################################################
connectivity = Connectivity()
connectivity.add_resonator_line(qubits=qubits, triggered=True, constraints=qubit_res_ch)
connectivity.add_qubit_drive_lines(qubits=qubits, triggered=True, constraints=qubit_xy_ch)
allocate_wiring(connectivity, instruments)

fig_wiring = visualize(connectivity.elements,
                        available_channels=instruments.available_channels)
plt.show(block=False)

# ## Build and save ########################################################
user_input = input("Save QUAM? (y/n) ").strip().lower()
if user_input == "y":
    machine = Quam()
    build_quam_wiring(connectivity, host_ip, cluster_name, machine)
    machine = Quam.load()
    build_quam(machine, calibration_db_path)

    # ## Output modes #####################################################
    # All active RF outputs use internal LO — no external source configuration needed.
    # RF_outputs/2 (f0g1) uses external LO but is added separately in the populate cell.
    for octave in machine.octaves.values():
        for rf_out in octave.RF_outputs.values():
            if rf_out.channel is not None:
                rf_out.output_mode = "triggered"

    # ## Loopbacks #########################################################
    # No loopbacks needed — resonator (RF1), qubit (RF3), and cavity (RF4) all
    # use the Octave's internal LO synthesizers directly.
    # f0g1 (RF2) uses an external LO connected to the Octave's LO2 input port.
    #
    # for oct_name, octave in machine.octaves.items():
    #     octave.loopbacks = [
    #         ((oct_name, "Synth1"), "Dmd2LO"),  # Synth1 -> RF_in2 down-conv LO
    #         ((oct_name, "Synth1"), "LO3"),      # Synth1 -> RF_out3 upconv LO
    #         ((oct_name, "Synth2"), "LO4"),      # Synth2 -> RF_out4 upconv LO
    #     ]

    machine.save()
    print("Done.  Add EF and cavity channels to state.json, then populate.")
else:
    print("Skipped.")

## 2. Populate QUAM with initial values

Edit the **USER PARAMETERS** section to match chip specs, then run.

In [ ]:
import json
import numpy as np
from pprint import pprint
from qualang_tools.units import unit
from quam.components.pulses import SquarePulse, DragCosinePulse, DragGaussianPulse
from quam.components.octave import OctaveUpConverter
from quam.components.channels import DigitalOutputChannel
from quam.components.ports import OPXPlusAnalogOutputPort, OPXPlusDigitalOutputPort
from quam_builder.architecture.superconducting.components.xy_drive import XYDriveIQ
from quam_builder.architecture.superconducting.cavity.cavity import Cavity
from quam_builder.architecture.superconducting.cavity.cavity_mode import CavityMode
from quam_builder.builder.superconducting.pulses import (
    add_DragGaussian_pulses,
)
from quam_config import Quam
from quam_builder.architecture.superconducting.qubit_pair import CavityTransmonPair

u = unit(coerce_to_integer=True)


def get_octave_gain_and_amplitude(desired_power: float, max_amplitude: float = 0.125):
    """Convert desired output power (dBm) to Octave gain + OPX IF amplitude."""
    octave_gain = round(max(min(desired_power - u.volts2dBm(max_amplitude), 20), -20) * 2) / 2
    amplitude = u.dBm2volts(desired_power - octave_gain)
    if not (-20 <= octave_gain <= 20 and -0.5 <= amplitude < 0.5):
        raise ValueError(f"Power outside spec: gain={octave_gain}, amp={amplitude}")
    return octave_gain, amplitude


machine = Quam.load()

##########################################################################
# USER PARAMETERS: edit to match your chip
##########################################################################
CAVITY_ID = "c1"    # key used in machine.cavities

# Hardware routing summary:
#   Resonator  → RF_outputs/1 (int LO synth1), OPX+ ports 1(I)/2(Q), trigger 1
#   f0g1       → RF_outputs/2 (ext LO 3 GHz),  OPX+ ports 3(I)/4(Q), trigger 3
#   Qubit XY   → RF_outputs/3 (int LO synth3),  OPX+ ports 5(I)/6(Q), trigger 5
#   Cavity     → RF_outputs/4 (int LO synth4),  OPX+ ports 7(I)/8(Q), trigger 7

rr_freq           = 7.504e9   # Hz  readout resonator frequency
rr_LO             = 7.400e9   # Hz  Octave RF_outputs/1 internal LO
readout_power     = 20        # dBm output power at cavity input
readout_gain      = 20        # dB  gain for input readout amplifiers

xy_freq           = 4.722e9   # Hz  qubit ge transition frequency
xy_LO             = 4.400e9   # Hz  Octave RF_outputs/3 internal LO
anharmonicity     = -200e6    # Hz  transmon anharmonicity (negative)
drive_power       = -10       # dBm qubit drive power (ge and ef use same amplitude)

alice_freq        = 6.000e9   # Hz  Alice cavity mode frequency
alice_LO          = 5.900e9   # Hz  Octave RF_outputs/4 internal LO (shared by alice + bob)
alice_power       = 20        # dBm cavity drive power
bob_freq          = 6.200e9   # Hz  Bob cavity mode frequency
bob_power         = 20        # dBm cavity drive power (same RF output as alice)

# ── f0g1 sideband drive (RF_outputs/2, ext LO 3 GHz, OPX+ 3/4 I/Q, trigger 3) ─
alice_f0g1_freq   = 3.25e9    # Hz  Initial estimate; refine after node 21
alice_f0g1_LO     = 3.0e9     # Hz  External LO frequency connected to LO2
alice_f0g1_gain   = 0          # dB  Octave RF_outputs/2 gain [-20, +20]
alice_f0g1_saturation_length_ns = 20000  # ns  long square saturation pulse for spectroscopy (node 21)
alice_f0g1_pi_length_ns     = 1000   # ns  Gaussian pi pulse length (calibrated by node 24)
alice_f0g1_sigma_ns          =  200   # ns  Gaussian sigma of f0g1_pi pulse (typically length/5)
alice_f0g1_amp    = 0.4        # V   initial f0g1 pulse amplitude (calibrated by nodes 22 / 24)
##########################################################################

readout_length_ns        = 8000
saturation_length_ns     = 20000
x180_length_ns           = 1000
gaussian_sigma_ns        = x180_length_ns // 5   # 200 ns for 1 µs pulse
drag_alpha               = 0.0   # DRAG alpha (tuned later by drag calibration)
drag_detuning            = 0.0   # DRAG detuning Hz (tuned later)
selective_x180_length_ns = 10000   # 10 µs → ~100 kHz bandwidth
f0g1_pulse_length_ns     = 1000
displacement_length_ns   = 1000   # ns  Gaussian displacement pulse length
displacement_sigma_ns    = displacement_length_ns // 5  # 200 ns Gaussian sigma (length/5)

T1 = 200e-6  # seconds
cavity_T1 = 100e-6  # seconds
resonator_depletion_time_ns = 10000
##########################################################################

assert abs(rr_freq - rr_LO) < 400e6,       'Resonator IF out of range'
assert abs(xy_freq - xy_LO) < 400e6,       'XY IF out of range'
assert abs(alice_freq - alice_LO) < 400e6, 'Alice IF out of range'
assert abs(bob_freq - alice_LO) < 400e6,   'Bob IF out of range (must share LO with Alice)'
assert abs(alice_f0g1_freq - alice_f0g1_LO) < 400e6, 'Alice f0g1 IF out of Octave range (must be <400 MHz)'

# ── Resonator hardware (OPX+ 1/2, digital 1, Octave RF1) ──────────────────────
_ao = machine.ports.analog_outputs.setdefault("con1", {})
_do = machine.ports.digital_outputs.setdefault("con1", {})

for port_id in (1, 2):
    if port_id not in _ao:
        _ao[port_id] = OPXPlusAnalogOutputPort(
            controller_id="con1", port_id=port_id, delay=0, shareable=False)

if 1 not in _do:
    _do[1] = OPXPlusDigitalOutputPort(
        controller_id="con1", port_id=1, shareable=False)

rr_rf1 = machine.octaves["oct1"].RF_outputs[1]
rr_rf1.LO_frequency = rr_LO
rr_rf1.LO_source    = "internal"
rr_rf1.output_mode  = "triggered"
if rr_rf1.gain is None:
    rr_rf1.gain = get_octave_gain_and_amplitude(readout_power, 0.125)[0]

# RF_inputs/2: down-converter shares LO with RF_outputs/1 (internal)
rr_rfi2 = machine.octaves["oct1"].RF_inputs[2]
rr_rfi2.LO_source    = "internal"
rr_rfi2.LO_frequency = "#/octaves/oct1/RF_outputs/1/LO_frequency"

# ── Qubit hardware (OPX+ 5/6, digital 5, Octave RF3) ──────────────────────────
for port_id in (5, 6):
    if port_id not in _ao:
        _ao[port_id] = OPXPlusAnalogOutputPort(
            controller_id="con1", port_id=port_id, delay=0, shareable=False)

if 5 not in _do:
    _do[5] = OPXPlusDigitalOutputPort(
        controller_id="con1", port_id=5, shareable=False)

xy_rf3 = machine.octaves["oct1"].RF_outputs[3]
xy_rf3.LO_frequency = xy_LO
xy_rf3.LO_source    = "internal"
xy_rf3.output_mode  = "triggered"
if xy_rf3.gain is None:
    xy_rf3.gain = get_octave_gain_and_amplitude(drive_power)[0]

# ── f0g1 sideband drive hardware (OPX+ 3/4 I/Q, digital 3, Octave RF2) ────────
for port_id in (3, 4):
    if port_id not in _ao:
        _ao[port_id] = OPXPlusAnalogOutputPort(
            controller_id="con1", port_id=port_id, delay=0, shareable=True)
    else:
        _ao[port_id].shareable = True

if 3 not in _do:
    _do[3] = OPXPlusDigitalOutputPort(
        controller_id="con1", port_id=3, shareable=False)

f0g1_rf2 = machine.octaves["oct1"].RF_outputs[2]
f0g1_rf2.LO_frequency = alice_f0g1_LO
f0g1_rf2.LO_source    = "external"   # RF2 uses an external LO
f0g1_rf2.gain         = alice_f0g1_gain
f0g1_rf2.output_mode  = "triggered"

# ── Cavity hardware (OPX+ 7/8, digital 7, Octave RF4) — alice + bob share ports
for port_id in (7, 8):
    if port_id not in _ao:
        _ao[port_id] = OPXPlusAnalogOutputPort(
            controller_id="con1", port_id=port_id, delay=0, shareable=True)
    else:
        _ao[port_id].shareable = True

if 7 not in _do:
    _do[7] = OPXPlusDigitalOutputPort(
        controller_id="con1", port_id=7, shareable=True)
else:
    _do[7].shareable = True

cav_rf4 = machine.octaves["oct1"].RF_outputs[4]
cav_gain, cav_amp = get_octave_gain_and_amplitude(alice_power)
cav_rf4.LO_frequency = alice_LO
cav_rf4.LO_source    = "internal"
cav_rf4.gain         = cav_gain
cav_rf4.output_mode  = "triggered"

# ── RF5: unused — keep internal LO, always off ────────────────────────────────
rf5 = machine.octaves["oct1"].RF_outputs[5]
rf5.LO_source   = "internal"
rf5.output_mode = "always_off"

# ── Loopbacks: cleared — all channels use internal Octave LOs except f0g1 ──────
machine.octaves["oct1"].loopbacks = []

# ── Resonator channel frequencies and pulses ──────────────────────────────────
rr_gain, rr_amp = get_octave_gain_and_amplitude(readout_power, 0.125)
for qubit in machine.qubits.values():
    qubit.resonator.f_01         = rr_freq
    qubit.resonator.RF_frequency = rr_freq
    qubit.resonator.frequency_converter_up.LO_frequency = rr_LO
    qubit.resonator.frequency_converter_up.gain         = rr_gain
    qubit.resonator.frequency_converter_up.output_mode  = "triggered"
    if qubit.resonator.depletion_time is None:
        qubit.resonator.depletion_time = resonator_depletion_time_ns
    # Pulse amplitudes: only set if not yet calibrated
    ro_op = qubit.resonator.operations.get('readout')
    if ro_op is not None and (ro_op.amplitude == 0 or ro_op.amplitude is None):
        ro_op.amplitude = rr_amp
    if ro_op is not None and ro_op.length == 0:
        ro_op.length = readout_length_ns

# ── Qubit XY channel frequencies and pulses ────────────────────────────────────
xy_gain, xy_amp = get_octave_gain_and_amplitude(drive_power)
for k, (q_name, qubit) in enumerate(machine.qubits.items()):
    qubit.f_01                                          = xy_freq
    qubit.xy.RF_frequency                               = xy_freq
    qubit.xy.frequency_converter_up.LO_frequency        = xy_LO
    qubit.xy.frequency_converter_up.gain                = xy_gain
    qubit.xy.frequency_converter_up.output_mode         = "triggered"
    if qubit.T1 is None:
        qubit.T1 = T1
    if qubit.anharmonicity is None:
        qubit.anharmonicity = int(anharmonicity)
    if qubit.grid_location is None or qubit.grid_location == '':
        qubit.grid_location = f"{k},0"
    # Pulse amplitudes and digital marker: only set if not yet calibrated
    sat_op = qubit.xy.operations.get('saturation')
    if sat_op is not None and (sat_op.amplitude == 0 or sat_op.amplitude is None):
        sat_op.amplitude = 0.3
    if sat_op is not None and sat_op.length == 0:
        sat_op.length = saturation_length_ns
    if sat_op is not None and sat_op.digital_marker is None:
        sat_op.digital_marker = "ON"
    add_DragGaussian_pulses(qubit, xy_amp, x180_length_ns, gaussian_sigma_ns,
                            drag_alpha, drag_detuning, anharmonicity,
                            digital_marker="ON")

# ── Cavity object (alice + bob) ───────────────────────────────────────────────
if CAVITY_ID not in machine.cavities:
    def _make_cavity_drive():
        drive = XYDriveIQ(
            opx_output_I="#/ports/analog_outputs/con1/7",
            opx_output_Q="#/ports/analog_outputs/con1/8",
            frequency_converter_up="#/octaves/oct1/RF_outputs/4",
            RF_frequency=None,
        )
        drive.digital_outputs["octave_switch_0"] = DigitalOutputChannel(
            opx_output="#/ports/digital_outputs/con1/7", delay=57, buffer=18,
        )
        return drive

    alice_mode = CavityMode(id="alice", cavity_mode_drive=_make_cavity_drive())
    alice_mode.T1 = cavity_T1
    bob_mode   = CavityMode(id="bob",   cavity_mode_drive=_make_cavity_drive())
    bob_mode.T1 = cavity_T1
    machine.cavities[CAVITY_ID] = Cavity(id=CAVITY_ID, alice=alice_mode, bob=bob_mode)
    cav_rf4.channel = f"#/cavities/{CAVITY_ID}/alice/cavity_mode_drive"
    print(f"  Created cavity '{CAVITY_ID}' with alice and bob modes.")
else:
    for mode_name in ("alice", "bob"):
        mode = getattr(machine.cavities[CAVITY_ID], mode_name, None)
        if mode is not None:
            mode.T1 = cavity_T1

# Cavity drive frequencies and pulses
for cav_name, cavity in machine.cavities.items():
    for mode_name, freq in (('alice', alice_freq), ('bob', bob_freq)):
        mode = getattr(cavity, mode_name, None)
        if mode is None or mode.cavity_mode_drive is None:
            continue
        mode.cavity_mode_drive.RF_frequency = freq
        if 'saturation' not in mode.cavity_mode_drive.operations:
            mode.cavity_mode_drive.operations['saturation'] = SquarePulse(
                length=readout_length_ns, amplitude=cav_amp, digital_marker='ON')
        if 'displacement' not in mode.cavity_mode_drive.operations:
            mode.cavity_mode_drive.operations['displacement'] = DragGaussianPulse(
                length=displacement_length_ns,
                amplitude=cav_amp,
                sigma=displacement_sigma_ns,
                alpha=0.0,
                anharmonicity=0,
                detuning=0.0,
                axis_angle=0,
                digital_marker="ON",
            )

# ── CavityTransmonPair (with sideband_drive) ──────────────────────────────────
for q_name in machine.qubits:
    for mode_name in ("alice", "bob"):
        pair_key = f"{q_name}_{mode_name}"
        if pair_key not in machine.cavity_transmon_pairs:
            machine.cavity_transmon_pairs[pair_key] = CavityTransmonPair(
                qubit_name=q_name, cavity_mode_name=mode_name
            )
            print(f"  Created CavityTransmonPair '{pair_key}'")

    # parity_time is calibrated by node 30; initialise to None on first run
    for pk in (f"{q_name}_alice", f"{q_name}_bob"):
        p = machine.cavity_transmon_pairs.get(pk)
        if p is not None and not hasattr(p, "parity_time"):
            p.parity_time = None

    alice_pair = machine.cavity_transmon_pairs.get(f"{q_name}_alice")
    if alice_pair is not None and alice_pair.sideband_drive is None:
        alice_drive = XYDriveIQ(
            id=f"{q_name}_alice_f0g1",
            opx_output_I="#/ports/analog_outputs/con1/3",
            opx_output_Q="#/ports/analog_outputs/con1/4",
            frequency_converter_up="#/octaves/oct1/RF_outputs/2",
            RF_frequency=alice_f0g1_freq,
        )
        alice_drive.digital_outputs["octave_switch_0"] = DigitalOutputChannel(
            opx_output="#/ports/digital_outputs/con1/3", delay=57, buffer=18,
        )
        alice_drive.operations["saturation"] = SquarePulse(
            length=alice_f0g1_saturation_length_ns,
            amplitude=alice_f0g1_amp,
            digital_marker="ON",
        )
        alice_drive.operations["f0g1_pi"] = DragCosinePulse(
            length=alice_f0g1_pi_length_ns,
            axis_angle=0.0,
            alpha=0.0,
            anharmonicity=0,
            amplitude=alice_f0g1_amp,
            digital_marker="ON",
        )
        alice_pair.sideband_drive = alice_drive
        f0g1_rf2.channel = f"#/cavity_transmon_pairs/{q_name}_alice/sideband_drive"
        print(f"  Created sideband_drive for '{q_name}_alice' on RF_outputs/2.")
    elif alice_pair is not None and alice_pair.sideband_drive is not None:
        alice_pair.sideband_drive.RF_frequency = alice_f0g1_freq

machine.save()
print('QUAM saved.')
with open('qua_config.json', 'w+') as f:
    json.dump(machine.generate_config(), f, indent=4)
print('QUA config saved.')

### 1b. Verify cavity state

`generate_quam_srf.py` creates `machine.cavities["c1"]` with `.alice` (and optionally `.bob`) 
CavityMode objects, each holding a `cavity_mode_drive` XYDriveIQ channel.  
Run the cell below to confirm the structure, then proceed to populate it with actual RF frequencies.

In [ ]:
from quam_config import Quam

machine = Quam.load()
print("Cavities:", list(machine.cavities.keys()))
cav = machine.cavities["c1"]
print("Alice cavity_mode_drive:", cav.alice.cavity_mode_drive)
print("Bob   cavity_mode_drive:", cav.bob.cavity_mode_drive if cav.bob else "(not set)")

## 3. Device setup

### 3a. Close other quantum machines

In [3]:
from qualibrate import QualibrationNode, NodeParameters
from quam_config import Quam

node = QualibrationNode[NodeParameters, Quam](
    name="00_close_other_qms",
    description="Close all other open QMs.",
    parameters=NodeParameters(),
)
node.machine = Quam.load()

@node.run_action()
def close_all_quantum_machines(node: QualibrationNode[NodeParameters, Quam]):
    qmm = node.machine.connect()
    qmm.close_all_qms()
    print("All quantum machines closed.")

2026-04-11 01:20:49,832 - qualibrate - INFO - Creating node 00_close_other_qms


Running action close_all_quantum_machines
2026-04-11 01:20:50,465 - qm - INFO     - Performing health check
2026-04-11 01:20:50,822 - qm - INFO     - Health check passed
All quantum machines closed.
Action close_all_quantum_machines finished


### 3b. Scope verification — all channels

Play each hardware element in an infinite loop to verify signals on the oscilloscope.
Edit the `ELEMENTS` dict to enable/disable individual channels. Run the **halt** cell below to stop.

In [ ]:
from qm import QuantumMachinesManager
from qm.qua import *
from quam_config import Quam

machine = Quam.load()
config = machine.generate_config()

qmm = QuantumMachinesManager(
    host=machine.network.host,
    cluster_name=machine.network.cluster_name,
)
qm = qmm.open_qm(config)

# ── Collect element names ──────────────────────────────────────────────────────
q = machine.qubits["q1"]
cav = machine.cavities.get("c1")
alice_pair = machine.cavity_transmon_pairs.get("q1_alice")

rr_el       = q.resonator.name
xy_el       = q.xy.name
alice_el    = cav.alice.cavity_mode_drive.name if cav else None
bob_el      = cav.bob.cavity_mode_drive.name   if cav else None
sideband_el = (alice_pair.sideband_drive.name
               if alice_pair is not None and alice_pair.sideband_drive is not None
               else None)

# ── Choose which elements to play (set False to skip) ─────────────────────────
ELEMENTS = {
    rr_el:       ("readout",    True),   # Resonator  — OPX 1/2, IF ~104 MHz
    xy_el:       ("saturation", True),   # Qubit XY   — OPX 5/6, IF ~322 MHz
    sideband_el: ("f0g1_pi",    True),   # f0g1 drive — OPX 3/4, IF ~250 MHz
    alice_el:    ("saturation", True),   # Alice cav  — OPX 7/8, IF ~100 MHz
    bob_el:      ("saturation", True),   # Bob cav    — OPX 7/8, IF ~300 MHz
}

active = [(el, op) for el, (op, en) in ELEMENTS.items() if en and el is not None]
print("Active elements:")
for el, op in active:
    print(f"  {el:40s}  op='{op}'")

with program() as scope_cw:
    with infinite_loop_():
        align(*[el for el, _ in active])
        for el, op in active:
            play(op, el)

print("\nPlaying in infinite loop — run the halt cell below to stop.")
scope_job = qm.execute(scope_cw)

In [23]:
scope_job.halt()

True

### 3c. Mixer calibration

In [12]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

mixer_cal = library.nodes["01a_mixer_calibration"].copy(name="mixer_calibration")
mixer_cal.parameters.qubits = ["q1"]
mixer_cal.run()

2026-04-02 18:06:19,584 - qualibrate - INFO - Creating node 01a_mixer_calibration
2026-04-02 18:06:19,661 - qualibrate - INFO - Copying node with name 01a_mixer_calibration with parameters name = 'mixer_calibration', node_parameters = {}
2026-04-02 18:06:19,671 - qualibrate - INFO - Creating node 01a_mixer_calibration
2026-04-02 18:06:19,732 - qualibrate - INFO - Run node mixer_calibration with parameters: {}


2026-04-02 18:06:19,848 - qm - INFO     - Performing health check
2026-04-02 18:06:20,420 - qm - INFO     - Health check passed
2026-04-02 18:06:23,169 - qm - INFO     - Opening QM
2026-04-02 18:06:23,169 - qm - INFO     - Calibrating q1.resonator
2026-04-02 18:06:25,567 - qm - INFO     - Compiling program
2026-04-02 18:06:31,890 - qm - INFO     - Calibrating q1.xy
2026-04-02 18:06:33,781 - qm - INFO     - Compiling program
2026-04-02 18:06:41,769 - qm - INFO     - Compiling program
2026-04-02 18:06:49,818 - qm - INFO     - Compiling program
2026-04-02 18:06:57,863 - qm - INFO     - Compiling program


c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qm\octave\_calibration_analysis.py:225: RuntimeWarning: invalid value encountered in sqrt
  _n = np.sqrt(_I2c * _Q2c)


2026-04-02 18:07:04,150 - qm - INFO     - Closing QM


2026-04-02 18:07:04,259 - qualibrate - INFO - Node mixer_calibration - Results for q1:  SUCCESS!
	resonator         -> LO leakage suppression: -26.9 dB | image rejection: -32.7 dB.
	xy_drive          -> LO leakage suppression: -48.4 dB | image rejection: -35.7 dB.

2026-04-02 18:07:04,263 - qualibrate - INFO - Node mixer_calibration - Results for alice:  SUCCESS!
	cavity_mode_drive -> LO leakage suppression: -16.7 dB | image rejection: -20.3 dB.

2026-04-02 18:07:04,267 - qualibrate - INFO - Node mixer_calibration - Results for bob:  SUCCESS!
	cavity_mode_drive -> LO leakage suppression: -20.3 dB | image rejection: -26.0 dB.

2026-04-02 18:07:04,270 - qualibrate - INFO - Node mixer_calibration - Results for q1_alice:  SUCCESS!
	sideband_drive        -> LO leakage suppression: -24.7 dB | image rejection: -42.5 dB.

c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qualang_tools\octave_tools\calibration_result_plotter.py:133: RuntimeWarning: invalid value encountered in sc

NodeRunSummary(name='mixer_calibration', description='\n    A simple program to calibrate Octave mixers for all qubits and resonators\n', created_at=datetime.datetime(2026, 4, 2, 18, 6, 19, 742549, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 4, 2, 18, 7, 9, 580566, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), parameters=Parameters(multiplexed=False, use_state_discrimination=False, reset_type='thermal', qubits=['q1'], calibrate_resonator=True, calibrate_drive=True, calibrate_cavity_drive=True, calibrate_sideband_drive=True, simulate=False, simulation_duration_ns=50000, use_waveform_report=True, timeout=120, load_data_id=None), outcomes={'q1': <Outcome.SUCCESSFUL: 'successful'>, 'alice': <Outcome.SUCCESSFUL: 'successful'>, 'bob': <Outcome.SUCCESSFUL: 'successful'>, 'q1_alice': <Outcome.SUCCESSFUL: 'successful'>}, error=None, initial_targets=['q1'], s

### 3d. Time of flight

In [ ]:
from quam_config import Quam
machine = Quam.load()
# Adjust TOF if needed:
machine.qubits['q1'].resonator.time_of_flight = 272 #24 + 280
machine.save()
print('TOF:', machine.qubits['q1'].resonator.time_of_flight)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

tof_node = library.nodes["01a_time_of_flight"].copy(name="time_of_flight")
tof_node.parameters.qubits = ["q1"]
tof_node.parameters.readout_amplitude_in_v = 0.01
tof_node.run()

2026-03-31 09:11:05,567 - qualibrate - INFO - Creating node 01a_time_of_flight
2026-03-31 09:11:05,657 - qualibrate - INFO - Copying node with name 01a_time_of_flight with parameters name = 'time_of_flight', node_parameters = {}
2026-03-31 09:11:05,667 - qualibrate - INFO - Creating node 01a_time_of_flight
2026-03-31 09:11:05,777 - qualibrate - INFO - Run node time_of_flight with parameters: {}


2026-03-31 09:11:05,999 - qm - INFO     - Performing health check
2026-03-31 09:11:06,620 - qm - INFO     - Health check passed
2026-03-31 09:11:09,184 - qm - INFO     - Opening QM
2026-03-31 09:11:09,204 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 09:11:09,304 - qm - INFO     - Executing program


2026-03-31 09:11:09,647 - qualibrate - INFO - Node time_of_flight - Execution report for job 1769103657677
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 0.09s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 0.16s
2026-03-31 09:11:09,656 - qm - INFO     - Closing QM


2026-03-31 09:11:09,738 - qualibrate - INFO - Node time_of_flight - Results for qubit q1:  SUCCESS!
	Time of flight to add: 0 ns
	Offsets to add for 'I': -59.2 mV & for Q: -0.0 mV

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\01a_time_of_flight.py:209: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 09:11:09,888 - qualibrate - INFO - Saving node time_of_flight to local storage
2026-03-31 09:11:10,256 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 09:11:10,278 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3109_time_of_flight_091109\quam_state


NodeRunSummary(name='time_of_flight', description='\n        TIME OF FLIGHT - OPX+ & LF-FEM\nThis sequence involves sending a readout pulse and capturing the raw ADC traces.\nThe data undergoes post-processing to calibrate three distinct parameters:\n    - Time of Flight: This represents the internal processing time and the propagation\n      delay of the readout pulse. Its value can be adjusted in the configuration under\n      "time_of_flight". This value is utilized to offset the acquisition window relative\n      to when the readout pulse is dispatched.\n\n    - Analog Inputs Offset: Due to minor impedance mismatches, the signals captured by\n      the OPX might exhibit slight offsets.\n\n    - Analog Inputs Gain: If a signal is constrained by digitization or if it saturates\n      the ADC, the variable gain of the OPX analog input, ranging from -12 dB to 20 dB,\n      can be modified to fit the signal within the ADC range of +/-0.5V.\n\nPrerequisites:\n    - Having initialized the

## 4. Readout resonator

### 4a. Wide resonator spectroscopy

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

broad_spec = library.nodes["02d_broad_resonator_spectroscopy"].copy(name="broad_res_spec")
broad_spec.parameters.qubits = ["q1"]
broad_spec.parameters.frequency_span_in_mhz = 300.0
broad_spec.parameters.frequency_step_in_mhz = 0.1
broad_spec.parameters.num_shots = 50
broad_spec.parameters.peak_prominence = 2.0
broad_spec.parameters.peak_width = (1, 10.0)
broad_spec.parameters.blacklist_exclusion_radius_mhz = 10.0
broad_spec.parameters.readout_power_dbm = 0.0
broad_spec.parameters.max_amp = 0.3
broad_spec.parameters.save_readout_amplitude = False
broad_spec.run()

2026-03-31 09:15:25,358 - qualibrate - INFO - Creating node 02d_broad_resonator_spectroscopy
2026-03-31 09:15:25,450 - qualibrate - INFO - Copying node with name 02d_broad_resonator_spectroscopy with parameters name = 'broad_res_spec', node_parameters = {}
2026-03-31 09:15:25,461 - qualibrate - INFO - Creating node 02d_broad_resonator_spectroscopy
2026-03-31 09:15:25,541 - qualibrate - INFO - Run node broad_res_spec with parameters: {}
2026-03-31 09:15:25,601 - qualibrate - INFO - Node broad_res_spec - Broad spectroscopy: temporarily set readout power to 0.0 dBm (max_amp=0.3)


Setting the Octave gain to 0.5 dB
Setting the readout amplitude to 0.298538261891796 V
2026-03-31 09:15:25,811 - qm - INFO     - Performing health check
2026-03-31 09:15:26,115 - qm - INFO     - Health check passed
2026-03-31 09:15:27,969 - qm - INFO     - Opening QM
2026-03-31 09:15:27,979 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 09:15:28,090 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 6.96s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 7.03s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 7.09s


2026-03-31 09:15:35,560 - qualibrate - INFO - Node broad_res_spec - Execution report for job 1769103657679
No errors


Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 7.14s
2026-03-31 09:15:35,570 - qm - INFO     - Closing QM


2026-03-31 09:15:35,620 - qualibrate - INFO - Node broad_res_spec - Results for qubit q1:  SUCCESS!
Detected resonator frequency: 7.554 GHz
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02d_broad_resonator_spectroscopy.py:217: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 09:15:35,860 - qualibrate - INFO - Saving node broad_res_spec to local storage
2026-03-31 09:15:36,225 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 09:15:36,247 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3110_broad_res_spec_091535\quam_state


NodeRunSummary(name='broad_res_spec', description="\n        1D BROAD-BAND RESONATOR SPECTROSCOPY\nThis sequence involves measuring the resonator by sending a readout pulse and demodulating the signals to extract the\n'I' and 'Q' quadratures across varying readout intermediate frequencies for all the active qubits.\nThe data is then post-processed to determine the resonator resonance frequency.\nThis frequency is used to update the readout frequency in the state.\n\nPrerequisites:\n    - Having calibrated the IQ mixer/Octave connected to the readout line (node 01a_mixer_calibration.py).\n    - Having calibrated the time of flight, offsets, and gains (node 01a_time_of_flight.py).\n    - Having initialized the QUAM state parameters for the readout pulse amplitude and duration, and the resonators depletion time.\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The readout frequency: qubit.resonator.f_01 & qubit.resonator.RF_frequency

### 4b. Resonator spectroscopy (fine)

In [12]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

res_spec = library.nodes["02a_resonator_spectroscopy"].copy(name="resonator_spec")
res_spec.parameters.qubits = ["q1"]
res_spec.parameters.frequency_span_in_mhz = 20.0
res_spec.parameters.frequency_step_in_mhz = 0.01
res_spec.parameters.readout_power_dbm = 20
res_spec.parameters.num_shots = 100
res_spec.run()

2026-03-30 23:41:19,843 - qualibrate - INFO - Creating node 02a_resonator_spectroscopy
2026-03-30 23:41:19,930 - qualibrate - INFO - Copying node with name 02a_resonator_spectroscopy with parameters name = 'resonator_spec', node_parameters = {}
2026-03-30 23:41:19,940 - qualibrate - INFO - Creating node 02a_resonator_spectroscopy
2026-03-30 23:41:20,020 - qualibrate - INFO - Run node resonator_spec with parameters: {}
2026-03-30 23:41:20,070 - qualibrate - INFO - Node resonator_spec - Resonator spectroscopy: temporarily set readout power to 20.0 dBm (max_amp=0.1)


Setting the Octave gain to 20 dB
Setting the readout amplitude to 0.31622776601683794 V
2026-03-30 23:41:20,261 - qm - INFO     - Performing health check
2026-03-30 23:41:20,571 - qm - INFO     - Health check passed
2026-03-30 23:41:22,595 - qm - INFO     - Opening QM
2026-03-30 23:41:22,607 - qm - INFO     - Sending program to QOP for compilation
2026-03-30 23:41:22,771 - qm - INFO     - Executing program


2026-03-30 23:41:25,450 - qualibrate - INFO - Node resonator_spec - Execution report for job 1769103657632
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 2.40s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 2.46s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 2.51s
2026-03-30 23:41:25,459 - qm - INFO     - Closing QM


2026-03-30 23:41:25,661 - qualibrate - INFO - Node resonator_spec - Results for qubit q1:  SUCCESS!
	Resonator frequency: 7.476 GHz | FWHM: 482.1 kHz | 
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02a_resonator_spectroscopy.py:221: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-30 23:41:25,852 - qualibrate - INFO - Saving node resonator_spec to local storage
2026-03-30 23:41:26,253 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-30 23:41:26,275 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-30\#3083_resonator_spec_234125\quam_state


NodeRunSummary(name='resonator_spec', description="\n        1D RESONATOR SPECTROSCOPY\nThis sequence involves measuring the resonator by sending a readout pulse and demodulating the signals to extract the\n'I' and 'Q' quadratures across varying readout intermediate frequencies for all the active qubits.\nThe data is then post-processed to determine the resonator resonance frequency.\nThis frequency is used to update the readout frequency in the state.\n\nPrerequisites:\n    - Having calibrated the IQ mixer/Octave connected to the readout line (node 01a_mixer_calibration.py).\n    - Having calibrated the time of flight, offsets, and gains (node 01a_time_of_flight.py).\n    - Having initialized the QUAM state parameters for the readout pulse amplitude and duration, and the resonators depletion time.\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The readout frequency: qubit.resonator.f_01 & qubit.resonator.RF_frequency\n", create

### 4c. Resonator punch-out (optimal readout power)

In [16]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

punch_out = library.nodes["02e_resonator_punch_out"].copy(name="resonator_punch_out")
punch_out.parameters.qubits = ["q1"]
punch_out.parameters.frequency_span_in_mhz = 100.0
punch_out.parameters.frequency_step_in_mhz = 1
punch_out.parameters.min_power_dbm = -40
punch_out.parameters.max_power_dbm = 0
punch_out.parameters.num_power_points = 2
punch_out.parameters.max_amp = 0.1
punch_out.parameters.num_shots = 200
punch_out.parameters.frequency_shift_threshold_in_hz = 1e6
punch_out.parameters.use_adaptive_span = False
punch_out.run()

2026-03-31 09:27:24,722 - qualibrate - INFO - Creating node 02e_resonator_punch_out
2026-03-31 09:27:24,811 - qualibrate - INFO - Copying node with name 02e_resonator_punch_out with parameters name = 'resonator_punch_out', node_parameters = {}


2026-03-31 09:27:24,821 - qualibrate - INFO - Creating node 02e_resonator_punch_out
2026-03-31 09:27:24,901 - qualibrate - INFO - Run node resonator_punch_out with parameters: {}


Setting the Octave gain to 10.0 dB
Setting the readout amplitude to 0.1 V
2026-03-31 09:27:25,141 - qm - INFO     - Performing health check
2026-03-31 09:27:25,454 - qm - INFO     - Health check passed
2026-03-31 09:27:27,646 - qm - INFO     - Opening QM
2026-03-31 09:27:27,666 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 09:27:27,946 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 16.18s


2026-03-31 09:27:44,487 - qualibrate - INFO - Node resonator_punch_out - Execution report for job 1769103657684
No errors


Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 16.25s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 16.31s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 16.36s
2026-03-31 09:27:44,501 - qm - INFO     - Closing QM


2026-03-31 09:27:44,580 - qualibrate - INFO - Node resonator_punch_out - Results for qubit q1:  SUCCESS!
Error code: SUCCESS (0)
Optimal readout power: -40.00 dBm | Resonator frequency: 7.532 GHz | (shift of 28.000 MHz)

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02e_resonator_punch_out.py:311: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 09:27:44,689 - qualibrate - INFO - Node resonator_punch_out - [q1] ERROR CODE: SUCCESS (0)
  CORRECTIVE ACTION: RESET_ADAPTIVE_PARAMS
  Updated state:
    Optimal power:          -40.00 dBm
    Low-power frequency:    7.504160 GHz
    Frequency shift:        28.000 MHz
2026-03-31 09:27:44,699 - qualibrate - INFO - Saving node resonator_punch_out to local storage


Setting the Octave gain to -20 dB
Setting the readout amplitude to 0.0316227766016838 V


2026-03-31 09:27:44,913 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 09:27:44,938 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3115_resonator_punch_out_092744\quam_state


NodeRunSummary(name='resonator_punch_out', description="\n        RESONATOR PUNCH-OUT SPECTROSCOPY\nThis sequence characterizes the resonator response as a function of readout power\nin order to detect power-induced shifts of the resonator frequency (punch-out).\nA readout pulse is applied and the demodulated 'I' and 'Q' quadratures are acquired\nfor all resonators simultaneously while sweeping the readout frequency at a small\nnumber of readout power levels.\n\nFor each power level, the resonator frequency is extracted directly from the\nmeasured response. By comparing the resonator frequency at low and high readout\npower, the presence of a power-induced frequency shift is detected. Based on this\nanalysis, an optimal readout power is selected that avoids resonator punch-out while\nmaintaining sufficient signal strength.\n\nPrerequisites:\n    - Having calibrated the resonator frequency at low power\n      (e.g., node 02a_resonator_spectroscopy.py).\n    - Having specified the desire

### 4c bis. Resonator spectroscopy vs power

In [7]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

res_spec_vs_power = library.nodes["02b_resonator_spectroscopy_vs_power"].copy(name="resonator_spectroscopy_vs_power")
res_spec_vs_power.parameters.max_power_dbm = -10
res_spec_vs_power.parameters.min_power_dbm = -40
res_spec_vs_power.parameters.num_power_points = 10
res_spec_vs_power.parameters.moving_average_filter_window_num_points = 1
res_spec_vs_power.parameters.derivative_smoothing_window_num_points = 1
res_spec_vs_power.parameters.frequency_span_in_mhz = 20
res_spec_vs_power.parameters.frequency_step_in_mhz = 0.1
res_spec_vs_power.parameters.num_shots = 200
res_spec_vs_power.run()

2026-03-30 23:35:49,634 - qualibrate - INFO - Creating node 02b_resonator_spectroscopy_vs_power
2026-03-30 23:35:49,742 - qualibrate - INFO - Copying node with name 02b_resonator_spectroscopy_vs_power with parameters name = 'resonator_spectroscopy_vs_power', node_parameters = {}
2026-03-30 23:35:49,752 - qualibrate - INFO - Creating node 02b_resonator_spectroscopy_vs_power
2026-03-30 23:35:49,832 - qualibrate - INFO - Run node resonator_spectroscopy_vs_power with parameters: {}


Setting the Octave gain to 0.0 dB
Setting the readout amplitude to 0.1 V
2026-03-30 23:35:50,092 - qm - INFO     - Performing health check
2026-03-30 23:35:50,403 - qm - INFO     - Health check passed
2026-03-30 23:35:52,528 - qm - INFO     - Opening QM
2026-03-30 23:35:52,538 - qm - INFO     - Sending program to QOP for compilation
2026-03-30 23:35:52,800 - qm - INFO     - Executing program


2026-03-30 23:35:57,842 - qualibrate - INFO - Node resonator_spectro... - Execution report for job 1769103657627
No errors


Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 4.74s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 4.79s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 4.84s
2026-03-30 23:35:57,852 - qm - INFO     - Closing QM


2026-03-30 23:35:57,993 - qualibrate - INFO - Node resonator_spectro... - Results for qubit q1:  FAIL!
Optimal readout power: -41.00 dBm | Resonator frequency: nan GHz | (shift of nan MHz)

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02b_resonator_spectroscopy_vs_power.py:228: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-30 23:35:58,093 - qualibrate - INFO - Saving node resonator_spectroscopy_vs_power to local storage
2026-03-30 23:35:58,353 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-30 23:35:58,381 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-30\#3078_resonator_spectroscopy_vs_power_233558\quam_state


NodeRunSummary(name='resonator_spectroscopy_vs_power', description="\n        RESONATOR SPECTROSCOPY VERSUS READOUT POWER\nThis sequence involves measuring the resonator by sending a readout pulse and\ndemodulating the signals to extract the 'I' and 'Q' quadratures for all resonators\nsimultaneously. This is done across various readout frequencies and amplitudes.\nBased on the results, one can determine if a qubit is coupled to the resonator by\nnoting the resonator frequency splitting. This information can then be used to adjust\nthe readout amplitude, choosing a readout amplitude value just before the observed\nfrequency splitting.\n\nPrerequisites:\n    - Having calibrated the resonator frequency (node 02a_resonator_spectroscopy.py).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The readout frequency at the optimal readout power: qubit.resonator.f_01 & qubit.resonator.RF_frequency\n    - The readout power: qubit.resonator.se

### 4d. Resonator spectroscopy at calibrated power

In [18]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

res_spec_lp = library.nodes["02a_resonator_spectroscopy"].copy(name="resonator_spec_low_power")
res_spec_lp.parameters.qubits = ["q1"]
res_spec_lp.parameters.frequency_span_in_mhz = 50.0
res_spec_lp.parameters.frequency_step_in_mhz = 0.05
res_spec_lp.parameters.num_shots = 100
res_spec_lp.parameters.readout_power_dbm = -45.0
res_spec_lp.parameters.max_amp = 0.1
res_spec_lp.parameters.save_readout_amplitude = True
res_spec_lp.run()

2026-03-31 14:18:51,136 - qualibrate - INFO - Creating node 02a_resonator_spectroscopy
2026-03-31 14:18:51,218 - qualibrate - INFO - Copying node with name 02a_resonator_spectroscopy with parameters name = 'resonator_spec_low_power', node_parameters = {}
2026-03-31 14:18:51,228 - qualibrate - INFO - Creating node 02a_resonator_spectroscopy
2026-03-31 14:18:51,308 - qualibrate - INFO - Run node resonator_spec_low_power with parameters: {}
2026-03-31 14:18:51,358 - qualibrate - INFO - Node resonator_spec_lo... - Resonator spectroscopy: temporarily set readout power to -45.0 dBm (max_amp=0.1)


Setting the Octave gain to -20 dB
Setting the readout amplitude to 0.01778279410038923 V
2026-03-31 14:18:51,519 - qm - INFO     - Performing health check
2026-03-31 14:18:51,829 - qm - INFO     - Health check passed
2026-03-31 14:18:54,235 - qm - INFO     - Opening QM
2026-03-31 14:18:54,249 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 14:18:54,370 - qm - INFO     - Executing program


2026-03-31 14:18:59,400 - qualibrate - INFO - Node resonator_spec_lo... - Execution report for job 1769103657749
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 4.72s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 4.76s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 4.81s
2026-03-31 14:18:59,409 - qm - INFO     - Closing QM


2026-03-31 14:18:59,510 - qualibrate - INFO - Node resonator_spec_lo... - Results for qubit q1:  SUCCESS!
	Resonator frequency: 7.503 GHz | FWHM: 1323.8 kHz | 
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\02a_resonator_spectroscopy.py:221: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 14:18:59,680 - qualibrate - INFO - Saving node resonator_spec_low_power to local storage


Setting the Octave gain to -20 dB
Setting the readout amplitude to 0.01778279410038923 V


2026-03-31 14:19:00,106 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 14:19:00,127 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3178_resonator_spec_low_power_141859\quam_state


NodeRunSummary(name='resonator_spec_low_power', description="\n        1D RESONATOR SPECTROSCOPY\nThis sequence involves measuring the resonator by sending a readout pulse and demodulating the signals to extract the\n'I' and 'Q' quadratures across varying readout intermediate frequencies for all the active qubits.\nThe data is then post-processed to determine the resonator resonance frequency.\nThis frequency is used to update the readout frequency in the state.\n\nPrerequisites:\n    - Having calibrated the IQ mixer/Octave connected to the readout line (node 01a_mixer_calibration.py).\n    - Having calibrated the time of flight, offsets, and gains (node 01a_time_of_flight.py).\n    - Having initialized the QUAM state parameters for the readout pulse amplitude and duration, and the resonators depletion time.\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The readout frequency: qubit.resonator.f_01 & qubit.resonator.RF_frequency\

### 4e. Readout depletion measurement

Measures how long the resonator takes to deplete photons after a readout pulse.
Sweeps the wait time `tau` between a first readout and a Ramsey sequence on the qubit.
Residual photons AC-Stark shift the qubit during the Ramsey idle time; fitting the
exponential decay gives the resonator depletion time constant.
Updates `qubit.resonator.depletion_time` to 3× the fitted time constant.

In [16]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ro_depletion = library.nodes["08c_readout_depletion"].copy(name="ro_depletion")
ro_depletion.parameters.qubits = ["q1"]
ro_depletion.parameters.min_wait_time_in_ns = 1000
ro_depletion.parameters.max_wait_time_in_ns = 50_000
ro_depletion.parameters.wait_time_num_points = 10
ro_depletion.parameters.log_or_linear_sweep = "linear"
ro_depletion.parameters.ramsey_idle_time_in_ns = 1000
ro_depletion.parameters.num_shots = 400
ro_depletion.run()

2026-03-13 22:51:05,432 - qualibrate - INFO - Creating node 08c_readout_depletion
2026-03-13 22:51:05,481 - qualibrate - INFO - Copying node with name 08c_readout_depletion with parameters name = 'ro_depletion', node_parameters = {}
2026-03-13 22:51:05,491 - qualibrate - INFO - Creating node 08c_readout_depletion
2026-03-13 22:51:05,571 - qualibrate - INFO - Run node ro_depletion with parameters: {}


2026-03-13 22:51:05,781 - qm - INFO     - Performing health check
2026-03-13 22:51:06,092 - qm - INFO     - Health check passed
2026-03-13 22:51:07,680 - qm - INFO     - Opening QM
2026-03-13 22:51:07,689 - qm - INFO     - Sending program to QOP for compilation
2026-03-13 22:51:08,041 - qm - INFO     - Executing program


2026-03-13 22:51:10,621 - qualibrate - INFO - Node ro_depletion - Execution report for job 1769103655788
No errors


Progress: [##################################################] 100.0% (n=400/400) --> elapsed time: 2.45s
Progress: [##################################################] 100.0% (n=400/400) --> elapsed time: 2.50s
2026-03-13 22:51:10,621 - qm - INFO     - Closing QM


2026-03-13 22:51:10,671 - qualibrate - INFO - Node ro_depletion - Depletion time for qubit q1: 38907 +/- 424867 ns --> FAIL!
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\08c_readout_depletion.py:207: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-13 22:51:10,761 - qualibrate - INFO - Saving node ro_depletion to local storage
2026-03-13 22:51:10,950 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-13 22:51:10,967 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-13\#2212_ro_depletion_225110\quam_state


NodeRunSummary(name='ro_depletion', description='\n        READOUT DEPLETION MEASUREMENT\n\nMeasures how long the resonator takes to deplete photons after a readout pulse.\n\nSequence (repeated n_shots times, sweeping tau):\n  1. First readout (excites resonator photons, result discarded)\n  2. Wait tau on resonator (photons decay)\n  3. Ramsey on qubit: x90 → wait(ramsey_idle_time) → x90\n  4. Second readout (measures qubit state)\n\nWhen tau is short, residual photons AC-Stark shift the qubit during the Ramsey\nidle time, changing the excited-state population. As tau grows the photons\ndeplete and the Ramsey outcome stabilises. Fitting the exponential decay gives\nthe resonator depletion time constant.\n\nPrerequisites:\n    - Calibrated readout parameters (nodes 02a, 02b).\n    - Calibrated x90 pulse (node 04b_power_rabi or 04c_time_rabi).\n    - Calibrated IQ blobs / rotation angle (node 07_iq_blobs).\n\nState update:\n    - qubit.resonator.depletion_time (in ns)\n', created_at=dat

## 5. Transmon ge calibration

### 5a. Qubit spectroscopy vs power


In [3]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

qubit_spec_vs_power = library.nodes["03c_qubit_spectroscopy_vs_power"].copy(name="qubit_spec_vs_power")
qubit_spec_vs_power.parameters.qubits = ["q1"]
qubit_spec_vs_power.parameters.frequency_span_in_mhz = 400.0
qubit_spec_vs_power.parameters.frequency_step_in_mhz = 1
qubit_spec_vs_power.parameters.min_power_dbm = -40.0
qubit_spec_vs_power.parameters.max_power_dbm = -10.0
qubit_spec_vs_power.parameters.num_power_points = 10
qubit_spec_vs_power.parameters.max_amplitude_opx = 0.1
qubit_spec_vs_power.parameters.min_amplitude_opx = 0.01
qubit_spec_vs_power.parameters.operation = "saturation"
qubit_spec_vs_power.parameters.operation_len_in_ns = 20_000
qubit_spec_vs_power.parameters.linewidth_threshold_hz = 2e6
qubit_spec_vs_power.parameters.power_buffer_db = 3.0
qubit_spec_vs_power.parameters.num_shots = 200
qubit_spec_vs_power.parameters.use_adaptive_span = False
qubit_spec_vs_power.parameters.signal_source = "I_rot"
qubit_spec_vs_power.run()

2026-03-31 14:01:19,943 - qualibrate - INFO - Creating node 03c_qubit_spectroscopy_vs_power
2026-03-31 14:01:20,036 - qualibrate - INFO - Copying node with name 03c_qubit_spectroscopy_vs_power with parameters name = 'qubit_spec_vs_power', node_parameters = {}
2026-03-31 14:01:20,046 - qualibrate - INFO - Creating node 03c_qubit_spectroscopy_vs_power
2026-03-31 14:01:20,147 - qualibrate - INFO - Run node qubit_spec_vs_power with parameters: {}


Setting the Octave gain to 0.0 dB
Setting the saturation amplitude to 0.1 V
2026-03-31 14:01:20,358 - qm - INFO     - Performing health check
2026-03-31 14:01:20,817 - qm - INFO     - Health check passed
2026-03-31 14:01:23,305 - qm - INFO     - Opening QM
2026-03-31 14:01:23,328 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 14:01:23,604 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 53.86s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 53.94s


2026-03-31 14:02:18,202 - qualibrate - INFO - Node qubit_spec_vs_power - Execution report for job 1769103657734
No errors


Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 54.01s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 54.10s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 54.15s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 54.21s
2026-03-31 14:02:18,212 - qm - INFO     - Closing QM


2026-03-31 14:02:18,347 - qualibrate - INFO - Node qubit_spec_vs_power - [q1] SUCCESS - Error code: OVER_SATURATED_SUCCESS (4)
  Selected power:  -36.33 dBm
  Qubit frequency: 4.723700 GHz
  Min linewidth:   3.87 MHz
  IW angle:        0.0071 rad
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\03c_qubit_spectroscopy_vs_power.py:334: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\03c_qubit_spectroscopy_vs_power.py:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[q1] Detected qubit frequency: 4.723700 GHz


C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\03c_qubit_spectroscopy_vs_power.py:347: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 14:02:19,163 - qualibrate - INFO - Node qubit_spec_vs_power - [q1] ERROR CODE: OVER_SATURATED_SUCCESS (4)
  CORRECTIVE ACTION: RESET_ADAPTIVE_PARAMS
  Updated state:
    XY power          = -36.33 dBm
    Octave gain       = -20.00 dB  (saved to temp_calibration)
    Pulse amplitude   = 0.1000
    Qubit frequency   = 4.723700 GHz
2026-03-31 14:02:19,163 - qualibrate - INFO - Saving node qubit_spec_vs_power to local storage


Setting the Octave gain to -20 dB
Setting the saturation amplitude to 0.048231784822393056 V
Setting the Octave gain to -20 dB
Setting the x180 amplitude to 0.048231784822393056 V


2026-03-31 14:02:20,000 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 14:02:20,022 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3163_qubit_spec_vs_power_140219\quam_state


NodeRunSummary(name='qubit_spec_vs_power', description='\n        QUBIT SPECTROSCOPY VS DRIVE POWER\nThis sequence involves probing the qubit transition by applying an XY drive while sweeping the drive power and\nintermediate frequency around the expected qubit transition for all active qubits.\nThe qubit response is measured via the readout resonator, and the demodulated I/Q signals are post-processed to extract\nthe qubit spectroscopy signal as a function of frequency and drive power.\n\nThe resulting 2D spectroscopy map is analyzed to identify the qubit transition frequency, assess power broadening\nand saturation effects, and select an appropriate drive power for subsequent calibrations.\nA rough estimate of the qubit frequency at the selected drive power is extracted and used to update the qubit state.\n\nPrerequisites:\n    - Having calibrated the IQ mixer/Octave connected to the XY control line (node 01a_mixer_calibration.py).\n    - Having calibrated the readout chain, includin

### 5b. Qubit spectroscopy

> **SRF note**: The qubit appears as a **peak**. Set `find_dip=False`.

In [26]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

qubit_spec = library.nodes["03a_qubit_spectroscopy"].copy(name="qubit_spec")
qubit_spec.parameters.qubits = ["q1"]
qubit_spec.parameters.find_dip = False              # qubit appears as peak in reflection readout
qubit_spec.parameters.frequency_span_in_mhz = 50.0
qubit_spec.parameters.frequency_step_in_mhz = 0.1
qubit_spec.parameters.operation = "saturation"
qubit_spec.parameters.operation_len_in_ns = 20_000
qubit_spec.parameters.operation_amplitude_factor = 1
qubit_spec.parameters.num_shots = 300
qubit_spec.parameters.signal_source = "I_rot"
qubit_spec.run()

2026-03-31 14:34:42,092 - qualibrate - INFO - Creating node 03a_qubit_spectroscopy
2026-03-31 14:34:42,184 - qualibrate - INFO - Copying node with name 03a_qubit_spectroscopy with parameters name = 'qubit_spec', node_parameters = {}
2026-03-31 14:34:42,194 - qualibrate - INFO - Creating node 03a_qubit_spectroscopy
2026-03-31 14:34:42,274 - qualibrate - INFO - Run node qubit_spec with parameters: {}


2026-03-31 14:34:42,476 - qm - INFO     - Performing health check
2026-03-31 14:34:42,787 - qm - INFO     - Health check passed
2026-03-31 14:34:45,079 - qm - INFO     - Opening QM
2026-03-31 14:34:45,088 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 14:34:45,310 - qm - INFO     - Executing program


2026-03-31 14:34:55,799 - qualibrate - INFO - Node qubit_spec - Execution report for job 1769103657757
No errors


Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 10.16s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 10.21s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 10.26s
2026-03-31 14:34:55,799 - qm - INFO     - Closing QM


2026-03-31 14:34:55,909 - qualibrate - INFO - Node qubit_spec - Results for qubit q1:  SUCCESS!
	Qubit frequency: 4.724 GHz | FWHM: 3068.2 kHz | The integration weight angle: 1.571 rad
 To get the desired FWHM, the saturation amplitude is updated to: 47.2 mV | To get the desired x180 gate, the x180 amplitude is updated to: 98.8 mV
 Residual chi2: 0.023
 
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\03a_qubit_spectroscopy.py:224: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 14:34:55,999 - qualibrate - INFO - Saving node qubit_spec to local storage
2026-03-31 14:34:56,204 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 14:34:56,228 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3186_qubit_spec_143456\quam_state


NodeRunSummary(name='qubit_spec', description='\n        QUBIT SPECTROSCOPY\nThis sequence involves sending a saturation pulse to the qubit, placing it in a mixed state,\nand then measuring the state of the resonator across various qubit drive frequencies.\nIn order to facilitate the qubit search, the qubit pulse duration and amplitude can be changed manually\nfrom the node parameters.\n\nThe data is post-processed to determine the qubit resonance frequency and the width of the peak.\n\nNote that it can happen that the qubit is excited by the image sideband or LO leakage instead of the desired sideband.\nThis is why calibrating the qubit mixer is highly recommended when using external mixers or the Octave.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The qubit 0->1 

### 5c. Time Rabi

Find the π-pulse duration by sweeping the qubit pulse length at fixed amplitude.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

time_rabi = library.nodes["04c_time_rabi"].copy(name="time_rabi")
time_rabi.parameters.qubits = ["q1"]
time_rabi.parameters.min_duration_ns = 16
time_rabi.parameters.max_duration_ns = 300
time_rabi.parameters.duration_step_ns = 4
time_rabi.parameters.num_shots = 200
time_rabi.parameters.operation_amplitude_factor = 1.0
# time_rabi.parameters.drive_power_dbm = 10.0  # optional: override XY power
time_rabi.run()

2026-03-31 14:09:30,028 - qualibrate - INFO - Creating node 04c_time_rabi
2026-03-31 14:09:30,118 - qualibrate - INFO - Copying node with name 04c_time_rabi with parameters name = 'time_rabi', node_parameters = {}
2026-03-31 14:09:30,118 - qualibrate - INFO - Creating node 04c_time_rabi


2026-03-31 14:09:30,208 - qualibrate - INFO - Run node time_rabi with parameters: {}
2026-03-31 14:09:30,259 - qualibrate - INFO - Node time_rabi - Time Rabi: temporarily set XY drive power to 10.0 dBm (max_amp=0.1)


Setting the Octave gain to 20.0 dB
Setting the x180 amplitude to 0.1 V
2026-03-31 14:09:30,418 - qm - INFO     - Performing health check
2026-03-31 14:09:30,888 - qm - INFO     - Health check passed
2026-03-31 14:09:33,420 - qm - INFO     - Opening QM
2026-03-31 14:09:33,430 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 14:09:33,716 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 14.72s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 14.79s


2026-03-31 14:09:48,789 - qualibrate - INFO - Node time_rabi - Execution report for job 1769103657742
No errors


Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 14.83s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 14.89s
2026-03-31 14:09:48,798 - qm - INFO     - Closing QM


2026-03-31 14:09:48,852 - qualibrate - INFO - Node time_rabi - Results for qubit q1:  FAIL!
	Pi-pulse duration: 4 ns | Chi2: 3996433885701301760.000 | Periods: 26.26
 
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\04c_time_rabi.py:219: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 14:09:48,934 - qualibrate - INFO - Node time_rabi - [q1] Reverted XY amplitude override (fit failed).
2026-03-31 14:09:48,934 - qualibrate - INFO - Saving node time_rabi to local storage
2026-03-31 14:09:49,076 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 14:09:49,094 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3171_time_rabi_140948\quam_state


NodeRunSummary(name='time_rabi', description='\n        TIME RABI\nThis sequence plays a qubit drive pulse with variable duration and measures the resonator\nfor different pulse durations.  The result is a Rabi oscillation in the I quadrature from\nwhich the π-pulse duration is extracted.\n\nPrerequisites:\n    - Having calibrated the IQ mixer/Octave connected to the qubit drive (node 01a).\n    - Having calibrated the readout (time of flight, offsets, gains).\n    - Having found the qubit frequency (node 03a_qubit_spectroscopy or 03c_qubit_spectroscopy_vs_power).\n\nState update:\n    - The qubit pulse duration for the selected operation:\n      qubit.xy.operations[operation].length  (in nanoseconds)\n    - If drive_power_dbm is set and the fit succeeds, the amplitude override\n      is kept in the state (not reverted). If the fit fails it is reverted.\n', created_at=datetime.datetime(2026, 3, 31, 14, 9, 30, 218784, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 

### 5d. Power Rabi

In [9]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

power_rabi = library.nodes["04b_power_rabi"].copy(name="power_rabi")
power_rabi.parameters.qubits = ["q1"]
power_rabi.parameters.min_amp_factor = 0.001
power_rabi.parameters.max_amp_factor = 1.5
power_rabi.parameters.amp_factor_step = 0.010
power_rabi.parameters.num_shots = 300
power_rabi.run()

2026-04-04 17:39:25,328 - qualibrate - INFO - Creating node 04b_power_rabi
2026-04-04 17:39:25,399 - qualibrate - INFO - Copying node with name 04b_power_rabi with parameters name = 'power_rabi', node_parameters = {}
2026-04-04 17:39:25,410 - qualibrate - INFO - Creating node 04b_power_rabi
2026-04-04 17:39:25,464 - qualibrate - INFO - Run node power_rabi with parameters: {}


2026-04-04 17:39:25,705 - qm - INFO     - Performing health check
2026-04-04 17:39:26,016 - qm - INFO     - Health check passed
2026-04-04 17:39:28,682 - qm - INFO     - Opening QM
2026-04-04 17:39:28,692 - qm - INFO     - Sending program to QOP for compilation
2026-04-04 17:39:28,831 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 30.15s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 30.23s


2026-04-04 17:39:59,473 - qualibrate - INFO - Node power_rabi - Execution report for job 1769103658058
No errors


Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 30.30s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 30.35s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 30.40s
2026-04-04 17:39:59,483 - qm - INFO     - Closing QM


2026-04-04 17:39:59,543 - qualibrate - INFO - Node power_rabi - Results for qubit q1:  SUCCESS!
The calibrated x180 amplitude: 120.38 mV (x1.00)
 Rabi periods in sweep: 0.73
 Residual chi2: 0.004
 
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\04b_power_rabi.py:234: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-04 17:39:59,694 - qualibrate - INFO - Saving node power_rabi to local storage
2026-04-04 17:40:00,050 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-04 17:40:00,064 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-04\#3445_power_rabi_173959\quam_state


NodeRunSummary(name='power_rabi', description='\n        POWER RABI WITH ERROR AMPLIFICATION\nThis sequence involves repeatedly executing the qubit pulse (such as x180) \'N\' times and\nmeasuring the state of the resonator across different qubit pulse amplitudes and number of pulses.\nBy doing so, the effect of amplitude inaccuracies is amplified, enabling a more precise measurement of the pi pulse\namplitude. The results are then analyzed to determine the qubit pulse amplitude suitable for the selected duration.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the qubit frequency (node 03a_qubit_spectroscopy.py).\n    - Having set the qubit gates duration (qubit.xy.operations["x180"].length).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The qubit pulse amplitude corresponding to the specified operation (x180, x90...)\n    (qubit.xy.operations[operation].amplitude)

### 5e. Ramsey (T2*)

In [5]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ramsey = library.nodes["06a_ramsey"].copy(name="ramsey")
ramsey.parameters.qubits = ["q1"]
ramsey.parameters.num_shots = 400
ramsey.parameters.frequency_detuning_in_mhz = 1
ramsey.parameters.max_wait_time_in_ns = 5_000
ramsey.parameters.wait_time_num_points = 100
ramsey.parameters.log_or_linear_sweep = "linear"
ramsey.run()

2026-04-07 23:17:12,688 - qualibrate - INFO - Creating node 06a_ramsey
2026-04-07 23:17:12,786 - qualibrate - INFO - Copying node with name 06a_ramsey with parameters name = 'ramsey', node_parameters = {}
2026-04-07 23:17:12,796 - qualibrate - INFO - Creating node 06a_ramsey
2026-04-07 23:17:12,876 - qualibrate - INFO - Run node ramsey with parameters: {}


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-04-07 23:17:13,216 - qm - INFO     - Performing health check
2026-04-07 23:17:13,521 - qm - INFO     - Health check passed
2026-04-07 23:17:16,448 - qm - INFO     - Opening QM
2026-04-07 23:17:16,457 - qm - INFO     - Sending program to QOP for compilation
2026-04-07 23:17:16,678 - qm - INFO     - Executing program


2026-04-07 23:17:39,879 - qualibrate - INFO - Node ramsey - Execution report for job 1769103658377
No errors


Progress: [##################################################] 100.0% (n=400/400) --> elapsed time: 22.92s
Progress: [##################################################] 100.0% (n=400/400) --> elapsed time: 22.98s
Progress: [##################################################] 100.0% (n=400/400) --> elapsed time: 23.03s
2026-04-07 23:17:39,889 - qm - INFO     - Closing QM
Action execute_qua_program finished
Running action analyse_data


C:\Users\td-srv-quantum\AppData\Roaming\Python\Python311\site-packages\xarray\computation\apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
2026-04-07 23:17:39,989 - qualibrate - INFO - Node ramsey - Results for qubit q1:  SUCCESS!
	Detuning to correct: -0.002 MHz | T2*: 13.3 µs
	Residual chi2: 0.005



...

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\06a_ramsey.py:226: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-07 23:17:40,129 - qualibrate - INFO - Saving node ramsey to local storage


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-04-07 23:17:40,294 - qualibrate - INFO - Saving machine state to db
2026-04-07 23:17:40,311 - qualibrate - WARNING - save failed: No database connection configured for project 'calib_1q'
2026-04-07 23:17:40,311 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-07 23:17:40,341 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-07\#3550_ramsey_231740\quam_state


Action save_results finished


NodeRunSummary(name='ramsey', description='\n        RAMSEY WITH VIRTUAL Z ROTATIONS\nThe program consists in playing a Ramsey sequence (x90 - idle_time - x90/y90 - measurement) for different idle times.\nInstead of detuning the qubit gates, the frame of the second x90 pulse is rotated (de-phased) to mimic an accumulated\nphase acquired for a given detuning after the idle time.\nThis method has the advantage of playing gates on resonance as opposed to the detuned Ramsey.\n\nFrom the results, one can fit the Ramsey oscillations and precisely measure the qubit resonance frequency and T2*.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated the qubit x180 pulse parameters (nodes 03a_qubit_spectroscopy.py and 04b_power_rabi.py).\n    - (optional) Having optimized the readout parameters (nodes 08a, 08b and 08c).\n    - Having specified the desired flux poi

### 5f. T1 (ge)

In [1]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

T1_ge = library.nodes["05_T1"].copy(name="T1_ge")
T1_ge.parameters.qubits = ["q1"]
T1_ge.parameters.num_shots = 500
T1_ge.parameters.min_wait_time_in_ns = 16
T1_ge.parameters.max_wait_time_in_ns = 300_000
T1_ge.parameters.wait_time_num_points = 100
T1_ge.parameters.log_or_linear_sweep = "linear"
T1_ge.run()

2026-04-13 10:12:07,877 - qm - INFO     - Starting session: 50425f03-6743-4621-bae1-f16f35a0ad95


2026-04-13 10:12:11,417 - qualibrate - WARNING - Getting calibration path from config
2026-04-13 10:12:11,427 - qualibrate - INFO - Scanning node file C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\00_close_other_qms.py
2026-04-13 10:12:11,497 - qualibrate - INFO - Creating node 00_close_other_qms
2026-04-13 10:12:12,029 - qualibrate - INFO - Scanning node file C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\00_hello_qua.py
c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(
2026-04-13 10:12:12,100 - qualibrate - INFO - Creating node 00_hello_qua
2026-04-13 10:12:12,140 - qualibrate - INFO - Scanning node file 

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-04-13 10:12:41,725 - qm - INFO     - Performing health check
2026-04-13 10:12:42,035 - qm - INFO     - Health check passed
2026-04-13 10:12:46,575 - qm - INFO     - Opening QM
2026-04-13 10:12:46,585 - qm - INFO     - Sending program to QOP for compilation
2026-04-13 10:12:46,716 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 30.15s


2026-04-13 10:13:17,190 - qualibrate - INFO - Node T1_ge - Execution report for job 1769103662022
No errors


Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 30.20s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 30.24s
2026-04-13 10:13:17,200 - qm - INFO     - Closing QM


...

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\05_T1.py:211: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-13 10:13:17,352 - qualibrate - INFO - Saving node T1_ge to local storage


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-04-13 10:13:17,596 - qualibrate - INFO - Saving machine state to db
2026-04-13 10:13:17,605 - qualibrate - WARNING - save failed: No database connection configured for project 'calib_1q'
2026-04-13 10:13:17,615 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-13 10:13:17,640 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-13\#3570_T1_ge_101317\quam_state


Action save_results finished


NodeRunSummary(name='T1_ge', description='\n        T1 MEASUREMENT\nThe sequence consists in putting the qubit in the excited stated by playing the x180 pulse and measuring the resonator\nafter a varying time. The qubit T1 is extracted by fitting the exponential decay of the measured quadratures/state.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated the qubit x180 pulse parameters (nodes 03a_qubit_spectroscopy.py and 04b_power_rabi.py).\n    - (optional) Having optimized the readout parameters (nodes 08a, 08b and 08c).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The T1 relaxation time: qubit.T1\n', created_at=datetime.datetime(2026, 4, 13, 10, 12, 41, 324907, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(202

### 5g. T1 Monitor (ge)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

T1_monitor = library.nodes["31_T1_monitor"].copy(name="T1_monitor_ge")
T1_monitor.parameters.qubits = ["q1"]
T1_monitor.parameters.n_iter = 5*50*8
T1_monitor.parameters.num_shots = 200
T1_monitor.parameters.min_wait_time_in_ns = 16
T1_monitor.parameters.max_wait_time_in_ns = 300_000
T1_monitor.parameters.wait_time_num_points = 71
T1_monitor.parameters.log_or_linear_sweep = "linear"
T1_monitor.run()

2026-04-10 01:10:40,736 - qualibrate - WARNING - Getting calibration path from config
2026-04-10 01:10:40,736 - qualibrate - INFO - Scanning node file C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\00_close_other_qms.py
2026-04-10 01:10:40,746 - qualibrate - INFO - Creating node 00_close_other_qms
2026-04-10 01:10:41,028 - qualibrate - INFO - Scanning node file C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\00_hello_qua.py
c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(
2026-04-10 01:10:41,099 - qualibrate - INFO - Creating node 00_hello_qua
2026-04-10 01:10:41,169 - qualibrate - INFO - Scanning node file 

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-04-10 01:11:08,407 - qm - INFO     - Performing health check
2026-04-10 01:11:09,009 - qm - INFO     - Health check passed
2026-04-10 01:11:13,102 - qm - INFO     - Opening QM
2026-04-10 01:11:13,112 - qm - INFO     - Sending program to QOP for compilation
2026-04-10 01:11:13,252 - qm - INFO     - Executing program
2026-04-10 01:11:19,713 - qm - INFO     - Closing QM


2026-04-10 01:11:19,773 - qualibrate - INFO - Node T1_monitor_ge - Iter 1/2000  |  t = 0.2 min  |  q1: T1 = 88.0 µs


2026-04-10 01:11:22,909 - qm - INFO     - Opening QM
2026-04-10 01:11:22,919 - qm - INFO     - Sending program to QOP for compilation
2026-04-10 01:11:23,071 - qm - INFO     - Executing program
2026-04-10 01:11:29,441 - qm - INFO     - Closing QM


2026-04-10 01:11:29,481 - qualibrate - INFO - Node T1_monitor_ge - Iter 2/2000  |  t = 0.3 min  |  q1: T1 = 78.0 µs


...

2026-04-10 01:12:36,741 - qualibrate - INFO - Node T1_monitor_ge - Iter 9/2000  |  t = 1.5 min  |  q1: T1 = 91.9 µs


2026-04-10 01:12:39,827 - qm - INFO     - Opening QM
2026-04-10 01:12:39,837 - qm - INFO     - Sending program to QOP for compilation
2026-04-10 01:12:40,028 - qm - INFO     - Executing program
2026-04-10 01:12:46,428 - qm - INFO     - Closing QM


2026-04-10 01:12:46,468 - qualibrate - INFO - Node T1_monitor_ge - Iter 10/2000  |  t = 1.6 min  |  q1: T1 = 98.9 µs


2026-04-10 01:12:49,808 - qm - INFO     - Opening QM
2026-04-10 01:12:49,828 - qm - INFO     - Sending program to QOP for compilation
2026-04-10 01:12:49,978 - qm - INFO     - Executing program
2026-04-10 01:12:56,415 - qm - INFO     - Closing QM


2026-04-10 01:12:56,446 - qualibrate - INFO - Node T1_monitor_ge - Iter 11/2000  |  t = 1.8 min  |  q1: T1 = 90.7 µs


### 5e. Spin echo (T2)

In [48]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

echo = library.nodes["06b_echo"].copy(name="T2_echo")
echo.parameters.qubits = ["q1"]
echo.parameters.num_shots = 300
echo.parameters.max_wait_time_in_ns = 15_000
echo.parameters.wait_time_num_points = 300
echo.parameters.log_or_linear_sweep = "linear"
echo.run()

2026-03-31 15:37:01,677 - qualibrate - INFO - Creating node 06b_echo
2026-03-31 15:37:01,756 - qualibrate - INFO - Copying node with name 06b_echo with parameters name = 'T2_echo', node_parameters = {}
2026-03-31 15:37:01,765 - qualibrate - INFO - Creating node 06b_echo
2026-03-31 15:37:01,846 - qualibrate - INFO - Run node T2_echo with parameters: {}


2026-03-31 15:37:02,270 - qm - INFO     - Performing health check
2026-03-31 15:37:02,570 - qm - INFO     - Health check passed
2026-03-31 15:37:04,860 - qm - INFO     - Opening QM
2026-03-31 15:37:04,870 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 15:37:05,885 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 64.92s


2026-03-31 15:38:11,513 - qualibrate - INFO - Node T2_echo - Execution report for job 1769103657779
No errors


Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 64.99s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 65.07s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 65.11s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 65.16s
2026-03-31 15:38:11,524 - qm - INFO     - Closing QM


C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\06b_echo.py:210: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 15:38:11,630 - qualibrate - INFO - Saving node T2_echo to local storage
2026-03-31 15:38:11,790 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 15:38:11,812 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3208_T2_echo_153811\quam_state


NodeRunSummary(name='T2_echo', description='\n        T2 echo MEASUREMENT\nThe sequence consists in playing an echo sequence (x90 - idle_time - x180 - idle_time - -x90 - measurement) for \ndifferent idle times.\nThe qubit T2 echo is extracted by fitting the exponential decay of the measured quadratures/state.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the qubit frequency precisely (node 06a_ramsey.py).\n    - (optional) Having optimized the readout parameters (nodes 08a, 08b and 08c).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nNext steps before going to the next node:\n    - Update the qubit T2 echo: qubit.T2echo.\n', created_at=datetime.datetime(2026, 3, 31, 15, 37, 1, 855682, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 3, 31, 15, 38, 11, 835026, tzinfo=datetime.timezone(datetime.timedelta

### 5f. IQ blobs

In [2]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

iq_blobs = library.nodes["07_iq_blobs"].copy(name="iq_blobs")
iq_blobs.parameters.qubits = ["q1"]
iq_blobs.parameters.num_shots = 4000
iq_blobs.run()

2026-04-10 01:18:05,608 - qualibrate - INFO - Creating node 07_iq_blobs
2026-04-10 01:18:05,718 - qualibrate - INFO - Copying node with name 07_iq_blobs with parameters name = 'iq_blobs', node_parameters = {}
2026-04-10 01:18:05,728 - qualibrate - INFO - Creating node 07_iq_blobs
2026-04-10 01:18:05,798 - qualibrate - INFO - Run node iq_blobs with parameters: {}


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-04-10 01:18:06,113 - qm - INFO     - Performing health check
2026-04-10 01:18:06,413 - qm - INFO     - Health check passed
2026-04-10 01:18:09,413 - qm - INFO     - Opening QM
2026-04-10 01:18:09,423 - qm - INFO     - Sending program to QOP for compilation
2026-04-10 01:18:09,664 - qm - INFO     - Executing program


2026-04-10 01:18:14,008 - qualibrate - INFO - Node iq_blobs - Execution report for job 1769103658467
No errors


Progress: [##################################################] 100.0% (n=4000/4000) --> elapsed time: 0.09s
Progress: [##################################################] 100.0% (n=4000/4000) --> elapsed time: 0.18s
2026-04-10 01:18:14,017 - qm - INFO     - Closing QM
Action execute_qua_program finished
Running action analyse_data


2026-04-10 01:18:14,210 - qualibrate - INFO - Node iq_blobs - Results for qubit q1:  SUCCESS!
IW angle: 2.4 deg | ge_threshold: 2.5 mV | rus_threshold: -3.6 mV | readout fidelity: 79.8 % 
 


...

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\07_iq_blobs.py:239: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-10 01:18:14,640 - qualibrate - INFO - Saving node iq_blobs to local storage


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-04-10 01:18:15,122 - qualibrate - INFO - Saving machine state to db
2026-04-10 01:18:15,137 - qualibrate - WARNING - save failed: No database connection configured for project 'calib_1q'
2026-04-10 01:18:15,137 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-10 01:18:15,163 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-10\#3561_iq_blobs_011814\quam_state


Action save_results finished


NodeRunSummary(name='iq_blobs', description='\n        IQ BLOBS\nThis sequence involves measuring the state of the resonator \'N\' times, first after thermalization (with the qubit in\nthe |g> state) and then after applying a x180 (pi) pulse to the qubit (bringing the qubit to the |e> state).\nThe resulting IQ blobs are displayed, and the data is processed to determine:\n    - The rotation angle required for the integration weights, ensuring that the\n      separation between |g> and |e> states aligns with the \'I\' quadrature.\n    - The threshold along the \'I\' quadrature for effective qubit state discrimination (at the center between the two blobs).\n    - The repeat-until-success threshold along the \'I\' quadrature for effective active reset (at the center of the |g> blob).\n    - The readout confusion matrix, which is also influenced by the x180 pulse fidelity.\n\nPrerequisites:\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated

### 5g. Readout frequency optimization

In [11]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ro_freq_opt = library.nodes["08a_readout_frequency_optimization"].copy(name="readout_freq_opt")
ro_freq_opt.parameters.qubits = ["q1"]
ro_freq_opt.parameters.frequency_span_in_mhz = 20.0
ro_freq_opt.parameters.frequency_step_in_mhz = 0.05
ro_freq_opt.parameters.num_shots = 200
ro_freq_opt.run()

2026-04-04 17:40:35,989 - qualibrate - INFO - Creating node 08a_readout_frequency_optimization
2026-04-04 17:40:36,052 - qualibrate - INFO - Copying node with name 08a_readout_frequency_optimization with parameters name = 'readout_freq_opt', node_parameters = {}
2026-04-04 17:40:36,052 - qualibrate - INFO - Creating node 08a_readout_frequency_optimization
2026-04-04 17:40:36,142 - qualibrate - INFO - Run node readout_freq_opt with parameters: {}


2026-04-04 17:40:36,343 - qm - INFO     - Performing health check
2026-04-04 17:40:36,753 - qm - INFO     - Health check passed
2026-04-04 17:40:39,977 - qm - INFO     - Opening QM
2026-04-04 17:40:39,987 - qm - INFO     - Sending program to QOP for compilation
2026-04-04 17:40:40,158 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 106.90s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 107.01s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 107.12s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 107.24s


2026-04-04 17:42:28,404 - qualibrate - INFO - Node readout_freq_opt - Execution report for job 1769103658060
No errors


Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 107.35s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 107.43s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 107.51s
2026-04-04 17:42:28,414 - qm - INFO     - Closing QM


2026-04-04 17:42:28,505 - qualibrate - INFO - Node readout_freq_opt - Results for qubit q1:  SUCCESS!
	Optimal readout frequency: 7.497 GHz (shifted by -6.20 MHz) | chi: -3.47 MHz

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\08a_readout_frequency_optimization.py:224: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-04 17:42:28,876 - qualibrate - INFO - Saving node readout_freq_opt to local storage
2026-04-04 17:42:29,483 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-04 17:42:29,503 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-04\#3447_readout_freq_opt_174228\quam_state


NodeRunSummary(name='readout_freq_opt', description="\n        READOUT OPTIMISATION: FREQUENCY\nThe sequence consists in measuring the state of the resonator after thermalization (qubit in |g>) and after\nplaying a pi pulse to the qubit (qubit in |e>) successively while sweeping the readout frequency.\nThe 'I' & 'Q' quadratures when the qubit is in |g> and |e> are extracted to derive the readout fidelity.\nThe optimal readout frequency is chosen as to maximize the state discrimination Signal-to-Noise Ratio (SNR).\n\nPrerequisites:\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated the qubit x180 pulse parameters (nodes 03a_qubit_spectroscopy.py and 04b_power_rabi.py).\n\nState update:\n    - The readout frequency: qubit.resonator.f_01 & qubit.resonator.RF_frequency\n    - The dispersive shift: qubit.chi\n", created_at=datetime.datetime(2026, 4, 4, 17, 40, 36, 142297, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 

### 5h. Readout length optimization

Finds the optimal readout pulse duration by maximising g/e discrimination fidelity.
Uses accumulated demodulation: IQ is accumulated in 16 ns chunks within a single pulse,
yielding fidelity vs cumulative readout length in one experiment.
Updates `qubit.resonator.operations["readout"].length`.

In [12]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ro_length_opt = library.nodes["08d_readout_length_optimization"].copy(name="ro_length_opt")
ro_length_opt.parameters.qubits = ["q1"]
ro_length_opt.parameters.num_shots = 2000
ro_length_opt.parameters.max_readout_length_in_ns = 16000
ro_length_opt.parameters.division_length_in_ns = 160
ro_length_opt.run()

2026-04-04 17:42:29,769 - qualibrate - INFO - Creating node 08d_readout_length_optimization
2026-04-04 17:42:29,833 - qualibrate - INFO - Copying node with name 08d_readout_length_optimization with parameters name = 'ro_length_opt', node_parameters = {}
2026-04-04 17:42:29,839 - qualibrate - INFO - Creating node 08d_readout_length_optimization
2026-04-04 17:42:29,919 - qualibrate - INFO - Run node ro_length_opt with parameters: {}


2026-04-04 17:42:30,241 - qm - INFO     - Performing health check
2026-04-04 17:42:30,822 - qm - INFO     - Health check passed
2026-04-04 17:42:34,021 - qm - INFO     - Opening QM
2026-04-04 17:42:34,031 - qm - INFO     - Sending program to QOP for compilation
2026-04-04 17:42:34,784 - qm - INFO     - Executing program
2026-04-04 17:42:34,957 - qm - WARNING  - Nothing to fetch: no results were found. Please wait until the results are ready.


2026-04-04 17:42:40,529 - qualibrate - INFO - Node ro_length_opt - Execution report for job 1769103658061
No errors


2026-04-04 17:42:40,536 - qm - INFO     - Closing QM######## ] 99.0% (n=1980/2000) --> elapsed time: 5.44s


2026-04-04 17:42:40,841 - qualibrate - INFO - Node ro_length_opt - Readout length for qubit q1: optimal = 4320 ns, fidelity = 8740.0% --> SUCCESS!
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\08d_readout_length_optimization.py:268: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-04 17:42:40,936 - qualibrate - INFO - Saving node ro_length_opt to local storage
2026-04-04 17:42:41,115 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-04 17:42:41,127 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-04\#3448_ro_length_opt_174240\quam_state


NodeRunSummary(name='ro_length_opt', description='\n        READOUT LENGTH OPTIMIZATION\n\nFinds the optimal readout pulse duration by maximising the g/e state discrimination fidelity.\n\nUses accumulated demodulation: within a single readout pulse the IQ signal is accumulated\nin chunks of `division_length_in_cc` clock cycles (= 4 ns each). For each shot both the\nground state (after thermalization) and the excited state (after x180) are measured. The\ntwo-state discriminator is applied at each cumulative length to compute fidelity vs time.\nThe readout pulse length is then updated to the length that gives the highest fidelity.\n\nNote: integration weight names ("rotated_cos", "rotated_sin", "rotated_minus_sin") can be\nadjusted via parameters if your readout operation uses different names.\n\nPrerequisites:\n    - Calibrated readout frequency and power (nodes 08a, 08b).\n    - Calibrated x180 pulse (node 04b or 04c).\n    - Calibrated IQ rotation angle (node 07_iq_blobs).\n\nState up

### 5h. Readout power optimization

In [9]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ro_pwr_opt = library.nodes["08b_readout_power_optimization"].copy(name="readout_power_opt")
ro_pwr_opt.parameters.qubits = ["q1"]
ro_pwr_opt.parameters.num_shots = 2000
ro_pwr_opt.parameters.start_amp = 0.5
ro_pwr_opt.parameters.end_amp = 1.5
ro_pwr_opt.parameters.num_amps = 10
ro_pwr_opt.run()

2026-04-05 19:04:27,249 - qualibrate - INFO - Creating node 08b_readout_power_optimization
2026-04-05 19:04:27,309 - qualibrate - INFO - Copying node with name 08b_readout_power_optimization with parameters name = 'readout_power_opt', node_parameters = {}
2026-04-05 19:04:27,319 - qualibrate - INFO - Creating node 08b_readout_power_optimization
2026-04-05 19:04:27,399 - qualibrate - INFO - Run node readout_power_opt with parameters: {}


2026-04-05 19:04:27,621 - qm - INFO     - Performing health check
2026-04-05 19:04:27,925 - qm - INFO     - Health check passed
2026-04-05 19:04:30,697 - qm - INFO     - Opening QM
2026-04-05 19:04:30,706 - qm - INFO     - Sending program to QOP for compilation
2026-04-05 19:04:30,897 - qm - INFO     - Executing program


2026-04-05 19:04:43,584 - qualibrate - INFO - Node readout_power_opt - Execution report for job 1769103658089
No errors


Progress: [##################################################] 100.0% (n=2000/2000) --> elapsed time: 0.09s
Progress: [##################################################] 100.0% (n=2000/2000) --> elapsed time: 0.18s
2026-04-05 19:04:43,594 - qm - INFO     - Closing QM


2026-04-05 19:04:46,205 - qualibrate - INFO - Node readout_power_opt - Results for qubit q1:  SUCCESS!
	Optimal readout amplitude: 20.461 mV

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\08b_readout_power_optimization.py:218: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-05 19:04:46,505 - qualibrate - INFO - Saving node readout_power_opt to local storage
2026-04-05 19:04:47,182 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-05 19:04:47,205 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-05\#3473_readout_power_opt_190446\quam_state


NodeRunSummary(name='readout_power_opt', description='\n        READOUT POWER OPTIMIZATION\nThe sequence consists in measuring the state of the resonator after thermalization (qubit in |g>) and after\nplaying a pi pulse to the qubit (qubit in |e>) successively while sweeping the readout amplitude.\nThe \'I\' & \'Q\' quadratures when the qubit is in |g> and |e> are extracted to derive the readout fidelity.\nThe optimal readout amplitude is chosen as to maximize the readout fidelity.\n\nPrerequisites:\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated the qubit x180 pulse parameters (nodes 03a_qubit_spectroscopy.py and 04b_power_rabi.py).\n\nState update:\n    - The readout amplitude: qubit.resonator.operations["readout"].amplitude\n    - The integration weight angle: qubit.resonator.operations["readout"].integration_weights_angle\n    - the ge discrimination threshold: qubit.resonator.operations["readout"].threshold\n    - the Repeat Un

## 6. Transmon ef calibration

### 6a. Qubit spectroscopy ef

> Reflection readout -> set `find_dip=True` here too.

In [56]:
# Add EF pulse operations to q1.xy if not already present.
# Only inserts the missing keys — all other state values are left unchanged.
from quam_config import Quam
from quam.components.pulses import DragGaussianPulse

machine = Quam.load()
xy = machine.qubits["q1"].xy

if "EF_x180" not in xy.operations:
    xy.operations["EF_x180"] = DragGaussianPulse(
        length=40,
        amplitude=0.1,
        sigma=8,
        alpha=0.0,
        anharmonicity=-200e6,
        detuning=0.0,
        subtracted=True,
        axis_angle=0,
        digital_marker="ON",
    )
    print("EF_x180 added")
else:
    print("EF_x180 already exists — skipped")

if "EF_x90" not in xy.operations:
    xy.operations["EF_x90"] = DragGaussianPulse(
        length="#../EF_x180/length",
        amplitude=0.05,
        sigma="#../EF_x180/sigma",
        alpha="#../EF_x180/alpha",
        anharmonicity="#../EF_x180/anharmonicity",
        detuning="#../EF_x180/detuning",
        subtracted="#../EF_x180/subtracted",
        axis_angle=0,
        digital_marker="#../EF_x180/digital_marker",
    )
    print("EF_x90 added")
else:
    print("EF_x90 already exists — skipped")

machine.save()
print("State saved.")

EF_x180 added
EF_x90 added
State saved.


In [75]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

qubit_spec_ef = library.nodes["12_qubit_spectroscopy_EF"].copy(name="qubit_spec_ef")
qubit_spec_ef.parameters.qubits = ["q1"]
qubit_spec_ef.parameters.find_dip = True
qubit_spec_ef.parameters.frequency_span_in_mhz = 300.0
qubit_spec_ef.parameters.frequency_step_in_mhz = 1
qubit_spec_ef.parameters.operation_len_in_ns = 10_000
qubit_spec_ef.parameters.operation_amplitude_factor = 0.1
qubit_spec_ef.parameters.num_shots = 100
qubit_spec_ef.run()

2026-03-31 17:42:03,071 - qualibrate - INFO - Creating node 12_qubit_spectroscopy_EF
2026-03-31 17:42:03,154 - qualibrate - INFO - Copying node with name 12_qubit_spectroscopy_EF with parameters name = 'qubit_spec_ef', node_parameters = {}
2026-03-31 17:42:03,164 - qualibrate - INFO - Creating node 12_qubit_spectroscopy_EF
2026-03-31 17:42:03,234 - qualibrate - INFO - Run node qubit_spec_ef with parameters: {}


2026-03-31 17:42:03,515 - qm - INFO     - Performing health check
2026-03-31 17:42:03,996 - qm - INFO     - Health check passed
2026-03-31 17:42:06,558 - qm - INFO     - Opening QM
2026-03-31 17:42:06,577 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 17:42:06,789 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 150.49s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 150.56s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 150.64s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 150.71s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 150.78s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 150.85s
Progress: [###################

2026-03-31 17:44:40,442 - qualibrate - INFO - Node qubit_spec_ef - Execution report for job 1769103657797
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 152.01s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 152.06s
2026-03-31 17:44:40,452 - qm - INFO     - Closing QM


2026-03-31 17:44:40,542 - qualibrate - INFO - Node qubit_spec_ef - Results for qubit q1:  SUCCESS!
	EF frequency: 4.586 GHz | FWHM: 8102.9 kHz | The integration weight angle: 0.690 rad
 To get the desired FWHM, the saturation amplitude is updated to: 11.6 mV | To get the desired EF_x180 gate, the EF_x180 amplitude is updated to: 30.4 mV
 Residual chi2: 0.225
 
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\12_Qubit_Spectroscopy_E_to_F.py:225: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 17:44:40,702 - qualibrate - ERROR - Failed to run node qubit_spec_ef
Traceback (most recent call last):
  File "c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qualibrate\qualibration_node.py", line 724, in run
    self.run_node_file(self.filepath)
  File "c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qualibrate\qualibration_node.py", line 766, in run_node_file
 

AttributeError: 'Parameters' object has no attribute 'update_integration_weights_angle'

### 6b. Time Rabi ef
Find the EF π-pulse duration by sweeping the EF drive pulse length.
A ge x180 prepares |e⟩ before the EF drive, and a final ge x180 improves readout fidelity.

In [57]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

time_rabi_ef = library.nodes["04d_time_rabi_ef"].copy(name="time_rabi_ef")
time_rabi_ef.parameters.qubits = ["q1"]
time_rabi_ef.parameters.min_duration_ns = 16
time_rabi_ef.parameters.max_duration_ns = 200
time_rabi_ef.parameters.duration_step_ns = 4
time_rabi_ef.parameters.num_shots = 200
time_rabi_ef.parameters.operation_amplitude_factor = 1.0
time_rabi_ef.parameters.ef_x180_operation = "EF_x180"
time_rabi_ef.run()

2026-03-31 16:02:07,072 - qualibrate - INFO - Creating node 04d_time_rabi_ef
2026-03-31 16:02:07,153 - qualibrate - INFO - Copying node with name 04d_time_rabi_ef with parameters name = 'time_rabi_ef', node_parameters = {}
2026-03-31 16:02:07,153 - qualibrate - INFO - Creating node 04d_time_rabi_ef
2026-03-31 16:02:07,243 - qualibrate - INFO - Run node time_rabi_ef with parameters: {}


2026-03-31 16:02:07,533 - qm - INFO     - Performing health check
2026-03-31 16:02:07,836 - qm - INFO     - Health check passed
2026-03-31 16:02:10,377 - qm - INFO     - Opening QM
2026-03-31 16:02:10,397 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 16:02:10,667 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 51.63s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 51.70s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 51.77s


2026-03-31 16:03:03,020 - qualibrate - INFO - Node time_rabi_ef - Execution report for job 1769103657785
No errors


Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 51.85s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 51.89s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 51.94s
2026-03-31 16:03:03,030 - qm - INFO     - Closing QM


2026-03-31 16:03:03,090 - qualibrate - INFO - Node time_rabi_ef - Results for qubit q1:  SUCCESS!
	EF pi-pulse duration: 24 ns | Chi2: 0.846 | Periods: 3.93
 
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\04d_time_rabi_ef.py:210: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 16:03:03,182 - qualibrate - INFO - Node time_rabi_ef - [q1] Updated EF_x180 duration: 24 ns
2026-03-31 16:03:03,192 - qualibrate - INFO - Saving node time_rabi_ef to local storage
2026-03-31 16:03:03,371 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 16:03:03,392 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3214_time_rabi_ef_160303\quam_state


NodeRunSummary(name='time_rabi_ef', description='\n        EF TIME RABI\nThis sequence prepares the qubit in |e⟩ via a ge x180 pulse, then plays the EF drive pulse\nwith a variable duration at the e→f transition frequency, and applies a final ge x180 before\nreadout for improved readout fidelity.\n\nThe result is a Rabi oscillation in the I quadrature from which the EF π-pulse duration is\nextracted.\n\nPrerequisites:\n    - Having calibrated the ge x180 pulse (nodes 03a, 04b/04c).\n    - Having found the EF transition frequency (node 12).\n    - Having a defined EF drive operation (e.g., EF_x180) in the QUAM state.\n\nState update:\n    - The EF pi-pulse duration: qubit.xy.operations[ef_x180_operation].length\n', created_at=datetime.datetime(2026, 3, 31, 16, 2, 7, 253257, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 3, 31, 16, 3, 3, 418586, tzinfo=datetime.timezone(datetime.timedelta(days=-1, secon

### 6c. Power Rabi ef

In [5]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

power_rabi_ef = library.nodes["13_power_rabi_ef"].copy(name="power_rabi_ef")
power_rabi_ef.parameters.qubits = ["q1"]
power_rabi_ef.parameters.min_amp_factor = 0.001
power_rabi_ef.parameters.max_amp_factor = 1.9
power_rabi_ef.parameters.amp_factor_step = 0.05
power_rabi_ef.parameters.num_shots = 200
power_rabi_ef.parameters.use_state_discrimination = True
power_rabi_ef.run()

2026-04-03 11:37:27,777 - qualibrate - INFO - Creating node 13_power_rabi_ef
2026-04-03 11:37:27,835 - qualibrate - INFO - Copying node with name 13_power_rabi_ef with parameters name = 'power_rabi_ef', node_parameters = {}
2026-04-03 11:37:27,838 - qualibrate - INFO - Creating node 13_power_rabi_ef
2026-04-03 11:37:27,911 - qualibrate - INFO - Run node power_rabi_ef with parameters: {}
2026-04-03 11:37:28,025 - qualibrate - ERROR - Failed to run node power_rabi_ef
Traceback (most recent call last):
  File "c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qualibrate\qualibration_node.py", line 724, in run
    self.run_node_file(self.filepath)
  File "c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qualibrate\qualibration_node.py", line 766, in run_node_file
    _module = import_from_path(
              ^^^^^^^^^^^^^^^^^
  File "c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qualibrate\utils\read_files.py", line 20, in import_from_path


TypeError: unsupported operand type(s) for +: 'float' and 'NoneType'

### 6d. Ramsey ef
Refines the EF transition frequency (corrects `q.anharmonicity`) and measures EF T2* via a virtual-Z Ramsey sequence on the e→f transition.

In [5]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ramsey_ef = library.nodes["06b_ramsey_ef"].copy(name="ramsey_ef")
ramsey_ef.parameters.qubits = ["q1"]
ramsey_ef.parameters.frequency_detuning_in_mhz = 1.0
ramsey_ef.parameters.min_wait_time_in_ns = 16
ramsey_ef.parameters.max_wait_time_in_ns = 3000
ramsey_ef.parameters.wait_time_num_points = 150
ramsey_ef.parameters.log_or_linear_sweep = "linear"
ramsey_ef.parameters.num_shots = 200
ramsey_ef.parameters.ef_x180_operation = "EF_x180"
ramsey_ef.run()

2026-03-31 18:05:19,812 - qualibrate - INFO - Creating node 06b_ramsey_ef
2026-03-31 18:05:19,880 - qualibrate - INFO - Copying node with name 06b_ramsey_ef with parameters name = 'ramsey_ef', node_parameters = {}
2026-03-31 18:05:19,889 - qualibrate - INFO - Creating node 06b_ramsey_ef
2026-03-31 18:05:19,979 - qualibrate - INFO - Run node ramsey_ef with parameters: {}


2026-03-31 18:05:20,411 - qm - INFO     - Performing health check
2026-03-31 18:05:20,842 - qm - INFO     - Health check passed
2026-03-31 18:05:23,176 - qm - INFO     - Opening QM
2026-03-31 18:05:23,186 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 18:05:23,316 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 301.05s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 301.13s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 301.20s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 301.28s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 301.35s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 301.43s
Progress: [###################

2026-03-31 18:10:27,671 - qualibrate - INFO - Node ramsey_ef - Execution report for job 1769103657807
No errors


Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 302.42s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 302.49s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 302.55s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 302.59s
2026-03-31 18:10:27,680 - qm - INFO     - Closing QM


C:\Users\td-srv-quantum\AppData\Roaming\Python\Python311\site-packages\xarray\computation\apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
2026-03-31 18:10:27,760 - qualibrate - INFO - Node ramsey_ef - Results for qubit q1:  SUCCESS!
	EF detuning to correct: -3.667 MHz | EF T2*: 3.6 µs
	Residual chi2: 0.036

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\06b_ramsey_ef.py:232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 18:10:27,920 - qualibrate - INFO - Saving node ramsey_ef to local storage
2026-03-31 18:10:28,097 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 18:10:28,117 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3235_ramsey_ef_181027\quam_state


NodeRunSummary(name='ramsey_ef', description='\n        EF RAMSEY WITH VIRTUAL Z ROTATIONS\nThe program prepares the qubit in |e⟩ via a ge π-pulse, then performs a Ramsey sequence\non the e→f transition: x90_ef – idle_time – x90_ef (with virtual detuning applied via\nframe rotation).  A final ge π-pulse is applied before readout to maximise readout contrast.\n\nThe EF Ramsey oscillation frequency is used to precisely determine the EF transition\nfrequency (i.e., correct the anharmonicity stored in the QUAM state), and the decay\nenvelope gives the EF coherence time T2*_ef.\n\nThe virtual detuning is applied symmetrically (± frequency_detuning_in_mhz) to\ndisambiguate the sign of the frequency correction.\n\nPrerequisites:\n    - Having calibrated the ge x180 and x90 pulses (nodes 03a, 04b/04c).\n    - Having run qubit EF spectroscopy to set q.anharmonicity (node 12).\n    - (optional) Having calibrated a dedicated x90_ef operation for better EF pi/2 pulses.\n\nState update:\n    - The 

### 6e. T1 of |f⟩ level

Measures the decay time of the second excited state (|f⟩ → |e⟩ relaxation).
Prepares |f⟩ via ge x180 + EF_x180, waits a variable idle time, then applies a final ge x180 before readout.

**State update**: `qubit.T1_ef`

> Prerequisite: calibrated `EF_x180` pulse (node 6c/6d).

In [2]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

T1_f = library.nodes["05b_T1_ef"].copy(name="T1_f")
T1_f.parameters.qubits = ["q1"]
T1_f.parameters.ef_x180_operation = "EF_x180"
T1_f.parameters.num_shots = 500
T1_f.parameters.min_wait_time_in_ns = 16
T1_f.parameters.max_wait_time_in_ns = 300_000
T1_f.parameters.wait_time_num_points = 100
T1_f.parameters.log_or_linear_sweep = "linear"
T1_f.run()

2026-03-31 21:59:08,249 - qualibrate - WARNING - Getting calibration path from config
2026-03-31 21:59:08,249 - qualibrate - INFO - Scanning node file C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\00_close_other_qms.py
2026-03-31 21:59:08,258 - qualibrate - INFO - Creating node 00_close_other_qms
2026-03-31 21:59:08,463 - qualibrate - INFO - Scanning node file C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\00_hello_qua.py
c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(
2026-03-31 21:59:08,503 - qualibrate - INFO - Creating node 00_hello_qua
2026-03-31 21:59:08,573 - qualibrate - INFO - Scanning node file 

2026-03-31 21:59:31,144 - qm - INFO     - Performing health check
2026-03-31 21:59:31,566 - qm - INFO     - Health check passed
2026-03-31 21:59:35,624 - qm - INFO     - Opening QM
2026-03-31 21:59:35,635 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 21:59:36,872 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 259.74s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 259.82s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 259.89s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 259.96s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 260.04s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 260.11s
Progress: [###################

2026-03-31 22:03:57,943 - qualibrate - INFO - Node T1_f - Execution report for job 1769103657809
No errors


Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 260.25s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 260.31s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 260.36s
2026-03-31 22:03:57,943 - qm - INFO     - Closing QM


2026-03-31 22:03:58,003 - qualibrate - INFO - Node T1_f - T1_ef for qubit q1: 182.96 ± 7.92 µs --> SUCCESS!
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\05b_T1_ef.py:200: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 22:03:58,093 - qualibrate - INFO - Saving node T1_f to local storage
2026-03-31 22:03:58,306 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 22:03:58,325 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3237_T1_f_220358\quam_state


NodeRunSummary(name='T1_f', description='\n        T1_ef MEASUREMENT\nThe sequence prepares the qubit in |f⟩ via two consecutive pi pulses (ge x180 then EF_x180),\nwaits a variable idle time, and then applies a final ge x180 before readout to improve\nreadout fidelity.  The exponential decay of the measured quadrature gives the |f⟩ lifetime T1_ef.\n\nThe signal decays from the f-state level (short t) to the e-state level (long t, |f⟩ → |e⟩\nrelaxation dominates).  The final ge x180 before readout maps |e⟩ → |g⟩ to exploit the\nbest-contrast readout state.\n\nPrerequisites:\n    - Having calibrated the ge x180 pulse (nodes 03a, 04b/04c).\n    - Having calibrated the EF_x180 pulse (node 13_power_rabi_ef).\n\nState update:\n    - The |f⟩ relaxation time: qubit.T1_ef\n', created_at=datetime.datetime(2026, 3, 31, 21, 59, 30, 683200, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 3, 31, 22, 3, 58, 343982, t

### 6f. GEF readout frequency optimization

Sweeps the readout IF around the current point while preparing the qubit in |g⟩, |e⟩,
and |f⟩. Finds the frequency that maximises the minimum centroid separation between all
three state pairs. Updates `qubit.resonator.GEF_frequency_shift`.

> Prerequisite: `EF_x180` operation calibrated (nodes 6b/6c).

In [11]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

gef_freq_opt = library.nodes["14_gef_frequency_optimization"].copy(name="gef_freq_opt")
gef_freq_opt.parameters.qubits = ["q1"]
gef_freq_opt.parameters.num_shots = 200
gef_freq_opt.parameters.frequency_span_in_mhz = 20.0
gef_freq_opt.parameters.frequency_step_in_mhz = 0.1
gef_freq_opt.run()

2026-04-01 10:06:01,172 - qualibrate - INFO - Creating node 14_gef_frequency_optimization
2026-04-01 10:06:01,262 - qualibrate - INFO - Copying node with name 14_gef_frequency_optimization with parameters name = 'gef_freq_opt', node_parameters = {}
2026-04-01 10:06:01,262 - qualibrate - INFO - Creating node 14_gef_frequency_optimization
2026-04-01 10:06:01,342 - qualibrate - INFO - Run node gef_freq_opt with parameters: {}


2026-04-01 10:06:01,764 - qm - INFO     - Performing health check
2026-04-01 10:06:02,382 - qm - INFO     - Health check passed
2026-04-01 10:06:04,946 - qm - INFO     - Opening QM
2026-04-01 10:06:04,956 - qm - INFO     - Sending program to QOP for compilation
2026-04-01 10:06:05,186 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 604.92s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 605.08s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 605.24s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 605.39s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 605.56s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 605.72s
Progress: [###################

2026-04-01 10:16:16,563 - qualibrate - INFO - Node gef_freq_opt - Execution report for job 1769103657841
No errors


2026-04-01 10:16:16,573 - qm - INFO     - Closing QM


2026-04-01 10:16:16,633 - qualibrate - INFO - Node gef_freq_opt - Results for qubit q1:  SUCCESS!
	Optimal frequency shift: 1.200 MHz | 
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\14_gef_readout_frequency_optimization.py:282: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-01 10:16:16,723 - qualibrate - INFO - Saving node gef_freq_opt to local storage
2026-04-01 10:16:16,923 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-01 10:16:16,943 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-01\#3266_gef_freq_opt_101616\quam_state


NodeRunSummary(name='gef_freq_opt', description="\n        G-E-F READOUT FREQUENCY OPTIMIZATION\nThis sequence sweeps the readout resonator intermediate frequency around the current operating point while preparing\nthe qubit successively in |g>, |e>, and |f> states. For every tested detuning, three IQ blobs (g, e, f) are acquired.\nThe distances between the three centroids are computed and fitted to identify the optimal frequency shift that\nmaximizes simultaneous separation (e.g. maximizes the minimum of {d_ge, d_ef, d_gf}). The resulting optimal detuning\nis then added to the stored `GEF_frequency_shift` parameter.\n\nPurpose:\n    - Optimize a single readout frequency for high-fidelity three-level (g/e/f) state discrimination\n        (including leakage monitoring).\n    - Improve discrimination robustness against slow frequency drifts or residual mis-calibration.\n\nMeasurement flow:\n    1. For each qubit, loop over the readout frequency detuning values.\n    2. For every detuning

### 6f-ii. GEF readout power optimisation

Sweeps the readout pulse amplitude for all three qubit states (|⟩g⟨, |⟩e⟨, |⟩f⟨) at the
GEF-optimised readout frequency (set by node 14). Computes
 vs amplitude and picks the maximum.

**State update**:  = optimal amplitude

> Prerequisites: GEF readout frequency calibrated (node 14); ge + EF π-pulses calibrated.

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

gef_power_opt = library.nodes["14b_readout_gef_power_optimization"].copy(name="gef_power_opt")
gef_power_opt.parameters.qubits = ["q1"]
gef_power_opt.parameters.num_shots = 200
gef_power_opt.parameters.min_amp_factor = 0.1
gef_power_opt.parameters.max_amp_factor = 1.9
gef_power_opt.parameters.num_amps = 30
gef_power_opt.run()

### 6f-iii. GEF readout length optimisation

Sweeps the cumulative readout integration time (via accumulated demodulation) for all
three qubit states (|g⟩, |e⟩, |f⟩) at the GEF-optimised frequency and power.
Computes  at each cumulative length and finds the optimum.

**State update**:  = optimal length [ns]

> Prerequisites: GEF readout frequency (node 14) and power (node 14b) calibrated;
> ge + EF π-pulses calibrated; integration weight names match parameters (default: iw1/iw2/iw3).

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

gef_length_opt = library.nodes["14c_readout_gef_length_optimization"].copy(name="gef_length_opt")
gef_length_opt.parameters.qubits = ["q1"]
gef_length_opt.parameters.num_shots = 2000
gef_length_opt.parameters.max_readout_length_in_ns = 4000
gef_length_opt.parameters.division_length_in_ns = 16
# gef_length_opt.parameters.cos_weight_name = "iw1"   # adjust if your integration weights differ
# gef_length_opt.parameters.sin_weight_name = "iw2"
# gef_length_opt.parameters.minus_sin_weight_name = "iw3"
gef_length_opt.run()

### 6g. GEF IQ blobs

Captures single-shot IQ blobs for all three states (|g⟩, |e⟩, |f⟩) at the optimised
GEF readout frequency. Plots IQ distributions and confusion matrix.
Updates `qubit.resonator.gef_centers` and `qubit.resonator.gef_confusion_matrix`.

In [12]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

gef_blobs = library.nodes["15_iq_blobs_gef"].copy(name="gef_blobs")
gef_blobs.parameters.qubits = ["q1"]
gef_blobs.parameters.num_shots = 2000
gef_blobs.parameters.operation = "readout"  # or "readout_QND"
gef_blobs.run()

2026-04-01 10:18:00,374 - qualibrate - INFO - Creating node 15_iq_blobs_gef
2026-04-01 10:18:00,450 - qualibrate - INFO - Copying node with name 15_iq_blobs_gef with parameters name = 'gef_blobs', node_parameters = {}
2026-04-01 10:18:00,450 - qualibrate - INFO - Creating node 15_iq_blobs_gef
2026-04-01 10:18:00,530 - qualibrate - INFO - Run node gef_blobs with parameters: {}


2026-04-01 10:18:01,010 - qm - INFO     - Performing health check
2026-04-01 10:18:01,592 - qm - INFO     - Health check passed
2026-04-01 10:18:04,224 - qm - INFO     - Opening QM
2026-04-01 10:18:04,234 - qm - INFO     - Sending program to QOP for compilation
2026-04-01 10:18:04,597 - qm - INFO     - Executing program


2026-04-01 10:18:35,657 - qualibrate - INFO - Node gef_blobs - Execution report for job 1769103657842
No errors


Progress: [##################################################] 100.0% (n=2000/2000) --> elapsed time: 0.10s
Progress: [##################################################] 100.0% (n=2000/2000) --> elapsed time: 0.20s
2026-04-01 10:18:35,667 - qm - INFO     - Closing QM


2026-04-01 10:18:35,747 - qualibrate - INFO - Node gef_blobs - GEF blobs for q1: SUCCESS | g:(-9.3,-3.0) mV | e:(-1.0,-7.5) mV | f:(-7.0,-5.9) mV | d_ge/σ=2.29, d_gf/σ=1.07, d_ef/σ=1.29
2026-04-01 10:18:35,747 - qualibrate - INFO - Node gef_blobs -   LDA fidelity: P(g|g)=0.739, P(e|e)=0.757, P(f|f)=0.446
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\15_iq_blobs_gef.py:256: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-01 10:18:35,949 - qualibrate - INFO - Saving node gef_blobs to local storage
2026-04-01 10:18:36,383 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-01 10:18:36,405 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-01\#3267_gef_blobs_101835\quam_state


NodeRunSummary(name='gef_blobs', description="\n        IQ BLOBS GEF\nThis sequence involves measuring the state of the resonator 'N' times, first after thermalization (with the qubit in\nthe |g> state), then after applying a x180 (pi) pulse to the qubit (bringing the qubit to the |e> state) and finally\nafter applying a x180 (pi) pulse plus an EF_180 pulse (bringing the qubit to the |f> state).\nThe resulting IQ blobs are displayed, and the data is processed to determine:\n    - The centers of the |g>, |e> and |f> state IQ blobs.\n    - The readout confusion matrix, which is also influenced by the x180 and EF_180 pulses fidelities.\n\nPrerequisites:\n    - Having calibrated the readout parameters (nodes 02a, 02b and/or 02c).\n    - Having calibrated the qubit x180 pulse parameters.\n    - Having calibrated the qubit EF_180 pulse parameters.\n\nState update:\n    - qubit.resonator.gef_centers (3×2 blob centres in raw ADC units, for readout_state_gef())\n    - qubit.gef_rotation_angle, 

### 6h. Qubit thermal population (RPM)

Measures the qubit thermal population P_th by comparing two EF Rabi sweeps:
- **'g' sweep** (from |g⟩): ge_π → ef(a) → ef_π → ge_π → readout  →  A_g
- **'e' sweep** (from thermal): ef(a) → ge_π → readout  →  A_e

P_th = A_e / (A_e + A_g).  Reports effective qubit temperature.  No state update.

In [3]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

rpm = library.nodes["20_qubit_rpm"].copy(name="qubit_rpm")
rpm.parameters.qubits = ["q1"]
rpm.parameters.num_shots = 500
rpm.parameters.min_amp_factor = -2.0
rpm.parameters.max_amp_factor = 1.99  # 2 full periods (np.arange stops before 4.0)
rpm.parameters.amp_factor_step = 0.02
rpm.run()


2026-04-11 01:02:15,842 - qualibrate - WARNING - Getting calibration path from config
2026-04-11 01:02:15,852 - qualibrate - INFO - Scanning node file C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\00_close_other_qms.py
2026-04-11 01:02:15,862 - qualibrate - INFO - Creating node 00_close_other_qms
2026-04-11 01:02:16,123 - qualibrate - INFO - Scanning node file C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\00_hello_qua.py
c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(
2026-04-11 01:02:16,183 - qualibrate - INFO - Creating node 00_hello_qua
2026-04-11 01:02:16,229 - qualibrate - INFO - Scanning node file 

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-04-11 01:02:45,405 - qm - INFO     - Performing health check
2026-04-11 01:02:45,705 - qm - INFO     - Health check passed
2026-04-11 01:02:50,122 - qm - INFO     - Opening QM


2026-04-11 01:02:50,122 - qualibrate - INFO - Node qubit_rpm - Running RPM 'g' sweep (start from |g⟩)…


2026-04-11 01:02:50,141 - qm - INFO     - Sending program to QOP for compilation
2026-04-11 01:02:50,342 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 92.26s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 92.31s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 92.36s


2026-04-11 01:04:23,128 - qualibrate - INFO - Node qubit_rpm - Running RPM 'e' sweep (start from thermal)…


...

2026-04-11 01:05:56,168 - qualibrate - INFO - Node qubit_rpm - RPM results for qubit q1: SUCCESS
	A_g = 0.2606 ± 0.0021
	A_e = 0.0278 ± 0.0017
	chi2_g = 0.006  (threshold 2.0)
	chi2_e = 0.322  (informational)
	P_th = (9.636 ± 0.524) %
	T_eff = (101.3 ± 2.7) mK
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\20_qubit_rpm.py:248: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-11 01:05:56,268 - qualibrate - INFO - Saving node qubit_rpm to local storage


Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-04-11 01:05:56,493 - qualibrate - INFO - Saving machine state to db
2026-04-11 01:05:56,508 - qualibrate - WARNING - save failed: No database connection configured for project 'calib_1q'
2026-04-11 01:05:56,508 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-11 01:05:56,536 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-11\#3565_qubit_rpm_010556\quam_state


Action save_results finished


NodeRunSummary(name='qubit_rpm', description="\n        QUBIT RABI POPULATION MEASUREMENT (RPM)\nMeasures the qubit thermal population by comparing two EF amplitude sweeps:\n\n  'g' sweep: ge_π → ef(a) → ef_π (back-swap) → ge_π → readout\n             Starts deterministically from |g⟩.\n             P_g(a) = cos²(π·a/2): oscillates 1→0→1, minimum at a=1.\n\n  'e' sweep: ef(a) → ge_π → readout\n             Starts from the thermal state. The |g⟩ component (1-P_th) always\n             maps to |e⟩ after ge_π; the |e⟩ component (P_th) undergoes EF\n             Rabi. P_e(a) = (1-P_th) + P_th·sin²(π·a/2).\n\nExtracting the sinusoidal amplitudes A_g and A_e:\n    P_th = A_e / (A_e + A_g)\n\nThe effective qubit temperature is derived from P_th and the qubit frequency.\n\nPrerequisites:\n    - Calibrated ge transition (node 07).\n    - Calibrated ef pulse: EF_x180 (node 13).\n\nState update:\n    None — this is a diagnostic node.\n", created_at=datetime.datetime(2026, 4, 11, 1, 2, 44, 745600,

### 6i. T1 & Thermal Population Monitor

Runs a T1 sweep followed by two RPM sweeps (start from |g⟩ and from thermal) in repeated iterations to monitor long-timescale correlations between T1 degradation and rising qubit thermal population.

Each iteration records T1 [µs] and P_th [%]. Results are saved as an xr.Dataset (.h5).

In [3]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

monitor = library.nodes["34_T1_thermal_monitor"].copy(name="T1_thermal_monitor")
monitor.parameters.qubits = ["q1"]
monitor.parameters.n_iter = 337
monitor.parameters.num_shots = 200
monitor.parameters.min_wait_time_in_ns = 16
monitor.parameters.max_wait_time_in_ns = 300_000
monitor.parameters.wait_time_num_points = 71
monitor.parameters.log_or_linear_sweep = "linear"
monitor.parameters.min_amp_factor = -2.0
monitor.parameters.max_amp_factor = 1.99
monitor.parameters.amp_factor_step = 0.02
monitor.run()

2026-04-12 12:53:01,815 - qualibrate - WARNING - Getting calibration path from config
2026-04-12 12:53:01,837 - qualibrate - INFO - Scanning node file C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\00_close_other_qms.py
2026-04-12 12:53:01,843 - qualibrate - INFO - Creating node 00_close_other_qms
2026-04-12 12:53:02,108 - qualibrate - INFO - Scanning node file C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\00_hello_qua.py
C:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\qm\results\__init__.py:15: DeprecationWarning: qm.results is deprecated since "1.2.3" and will be removed in "2.0.0". If you need anything from this module, import it directly from `qm` or from `qm.simulate` for simulator-related functionality.
  warnings.warn(
2026-04-12 12:53:02,187 - qualibrate - INFO - Creating node 00_hello_qua
2026-04-12 12:53:02,238 - qualibrate - INFO - Scanning node file 

Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-04-12 12:53:29,744 - qm - INFO     - Performing health check
2026-04-12 12:53:30,163 - qm - INFO     - Health check passed
2026-04-12 12:53:34,514 - qm - INFO     - Opening QM
2026-04-12 12:53:34,524 - qm - INFO     - Sending program to QOP for compilation
2026-04-12 12:53:34,670 - qm - INFO     - Executing program
2026-04-12 12:53:43,493 - qm - INFO     - Closing QM
2026-04-12 12:53:46,795 - qm - INFO     - Opening QM
2026-04-12 12:53:46,812 - qm - INFO     - Sending program to QOP for compilation
2026-04-12 12:53:46,941 - qm - INFO     - Executing program
2026-04-12 12:54:23,900 - qm - INFO     - Closing QM
2026-04-12 12:54:26,691 - qm - INFO     - Opening QM
2026-04-12 12:54:26,704 - qm - INFO     - Sending program to QOP for compilation
2026-04-12 12:54:26,900 - qm - INFO     - Executing program
2026-04-12 12:55:03,831 - qm - INFO     - Closing QM


2026-04-12 12:55:03,886 - qualibrate - INFO - Node T1_thermal_monitor - Iter 1/337  t=1.6 min  |  q1: T1=96.5µs  P_th=(6.348±0.742)%


2026-04-12 12:55:06,628 - qm - INFO     - Opening QM
2026-04-12 12:55:06,641 - qm - INFO     - Sending program to QOP for compilation
2026-04-12 12:55:06,755 - qm - INFO     - Executing program
2026-04-12 12:55:15,575 - qm - INFO     - Closing QM
2026-04-12 12:55:18,733 - qm - INFO     - Opening QM
2026-04-12 12:55:18,751 - qm - INFO     - Sending program to QOP for compilation
2026-04-12 12:55:18,931 - qm - INFO     - Executing program
2026-04-12 12:55:55,815 - qm - INFO     - Closing QM
2026-04-12 12:55:58,565 - qm - INFO     - Opening QM
2026-04-12 12:55:58,575 - qm - INFO     - Sending program to QOP for compilation
2026-04-12 12:55:58,756 - qm - INFO     - Executing program
2026-04-12 12:56:35,682 - qm - INFO     - Closing QM


2026-04-12 12:56:35,732 - qualibrate - INFO - Node T1_thermal_monitor - Iter 2/337  t=3.1 min  |  q1: T1=78.7µs  P_th=(7.522±0.695)%


...

2026-04-12 20:14:24,280 - qualibrate - INFO - Node T1_thermal_monitor - Iter 288/337  t=440.9 min  |  q1: T1=100.8µs  P_th=(4.730±0.469)%


2026-04-12 20:14:27,029 - qm - INFO     - Opening QM
2026-04-12 20:14:27,039 - qm - INFO     - Sending program to QOP for compilation
2026-04-12 20:14:27,149 - qm - INFO     - Executing program
2026-04-12 20:14:36,044 - qm - INFO     - Closing QM
2026-04-12 20:14:39,112 - qm - INFO     - Opening QM
2026-04-12 20:14:39,122 - qm - INFO     - Sending program to QOP for compilation
2026-04-12 20:14:39,292 - qm - INFO     - Executing program
2026-04-12 20:15:16,151 - qm - INFO     - Closing QM
2026-04-12 20:15:18,976 - qm - INFO     - Opening QM
2026-04-12 20:15:18,986 - qm - INFO     - Sending program to QOP for compilation
2026-04-12 20:15:19,108 - qm - INFO     - Executing program
2026-04-12 20:15:56,104 - qm - INFO     - Closing QM


2026-04-12 20:15:56,155 - qualibrate - INFO - Node T1_thermal_monitor - Iter 289/337  t=442.4 min  |  q1: T1=112.0µs  P_th=(4.086±0.449)%


2026-04-12 20:15:58,934 - qm - INFO     - Opening QM
2026-04-12 20:15:58,952 - qm - INFO     - Sending program to QOP for compilation
2026-04-12 20:15:59,135 - qm - INFO     - Executing program
2026-04-12 20:16:07,934 - qm - INFO     - Closing QM
2026-04-12 20:16:11,007 - qm - INFO     - Opening QM
2026-04-12 20:16:11,017 - qm - INFO     - Sending program to QOP for compilation
2026-04-12 20:16:11,157 - qm - INFO     - Executing program
2026-04-12 20:16:48,063 - qm - INFO     - Closing QM


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



### 6i. EF Rabi RPM — f-state preparation check

Calibrates the EF π-pulse amplitude using a back-swap readout scheme (no GEF readout required).  The signal P(a) = cos²(π·a/2) has a minimum at a=1 (correct π pulse).

**Sequence**: ge_π → ef(a) → ef_π (back-swap) → ge_π → readout

**State update**: `q1.xy.operations["EF_x180"].amplitude`

In [22]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

ef_rpm = library.nodes["20b_ef_rabi_rpm"].copy(name="ef_rabi_rpm")
ef_rpm.parameters.qubits = ["q1"]
ef_rpm.parameters.num_shots = 50
ef_rpm.parameters.min_amp_factor = 0.0
ef_rpm.parameters.max_amp_factor = 1.99
ef_rpm.parameters.amp_factor_step = 0.02
ef_rpm.parameters.use_state_discrimination = True
ef_rpm.run()


2026-04-04 18:09:28,469 - qualibrate - INFO - Creating node 20b_ef_rabi_rpm
2026-04-04 18:09:28,540 - qualibrate - INFO - Copying node with name 20b_ef_rabi_rpm with parameters name = 'ef_rabi_rpm', node_parameters = {}
2026-04-04 18:09:28,540 - qualibrate - INFO - Creating node 20b_ef_rabi_rpm
2026-04-04 18:09:28,630 - qualibrate - INFO - Run node ef_rabi_rpm with parameters: {}


2026-04-04 18:09:28,910 - qm - INFO     - Performing health check
2026-04-04 18:09:29,211 - qm - INFO     - Health check passed
2026-04-04 18:09:31,868 - qm - INFO     - Opening QM
2026-04-04 18:09:31,878 - qm - INFO     - Sending program to QOP for compilation
2026-04-04 18:09:32,008 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 26.02s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 26.07s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 26.12s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 26.17s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 26.22s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 26.27s
Progress: [#####################################

2026-04-04 18:09:59,331 - qualibrate - INFO - Node ef_rabi_rpm - Execution report for job 1769103658071
No errors


Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 26.52s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 26.57s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 26.61s
Progress: [##################################################] 100.0% (n=50/50) --> elapsed time: 26.64s
2026-04-04 18:09:59,340 - qm - INFO     - Closing QM


2026-04-04 18:09:59,380 - qualibrate - INFO - Node ef_rabi_rpm - EF Rabi RPM results for qubit q1: SUCCESS
	EF π-amp factor = 0.874 (ideal = 1.000)  →  amplitude = 60.72 mV
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\20b_ef_rabi_rpm.py:220: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-04 18:09:59,460 - qualibrate - INFO - Saving node ef_rabi_rpm to local storage
2026-04-04 18:09:59,630 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-04 18:09:59,642 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-04\#3458_ef_rabi_rpm_180959\quam_state


NodeRunSummary(name='ef_rabi_rpm', description='\n        EF RABI RPM — f-STATE PREPARATION CHECK\nUses the RPM back-swap readout scheme to calibrate the EF π-pulse amplitude\nwhile relying only on standard ge state discrimination (no GEF readout needed).\n\nSequence:\n  1. Qubit thermalization wait\n  2. ge_π  →  |e⟩\n  3. ef(a) [sweep amplitude]  →  EF Rabi rotation\n  4. ef_π  (back-swap: maps |f⟩→|e⟩ and |e⟩→|f⟩)\n  5. ge_π  (maps |e⟩→|g⟩; |f⟩ stays as |f⟩ → detected as excited)\n  6. Readout\n\nSignal: P(a) = cos²(π·a/2) — starts at 1, minimum at a=1 (correct π pulse),\nreturns to 1 at a=2.  The first minimum gives the EF π-pulse amplitude.\n\nWhy use this instead of direct EF power Rabi?\n- No need for GEF-optimized readout.\n- Back-swap readout gives full contrast (0→1) using only ge discrimination.\n- Directly validates that |f⟩ state preparation is correct end-to-end.\n\nPrerequisites:\n    - Calibrated ge transition (node 07).\n    - Rough EF pulse: EF_x180 (node 13).\n\nStat

## 7. Dispersive shift (chi)

Measures chi = f_resonator(|e>) - f_resonator(|g>) and sets the optimal readout frequency at the mid-point for maximum contrast.

In [5]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

disp_shift = library.nodes["20_dispersive_shift"].copy(name="dispersive_shift")
disp_shift.parameters.qubits = ["q1"]
disp_shift.parameters.num_shots = 200
disp_shift.parameters.frequency_span_in_mhz = 30.0    # total span of the frequency sweep [MHz]
disp_shift.parameters.frequency_step_in_mhz = 0.05   # frequency step size [MHz]
disp_shift.parameters.min_dip_contrast = 0.05         # minimum contrast to declare a dip found
disp_shift.parameters.lo_leakage_exclusion_mhz = 10.0 # frequency window around LO to exclude [MHz]
disp_shift.run()

2026-03-31 22:42:06,376 - qualibrate - INFO - Creating node 20_dispersive_shift
2026-03-31 22:42:06,458 - qualibrate - INFO - Copying node with name 20_dispersive_shift with parameters name = 'dispersive_shift', node_parameters = {}
2026-03-31 22:42:06,468 - qualibrate - INFO - Creating node 20_dispersive_shift
2026-03-31 22:42:06,559 - qualibrate - INFO - Run node dispersive_shift with parameters: {}


2026-03-31 22:42:06,929 - qm - INFO     - Performing health check
2026-03-31 22:42:07,231 - qm - INFO     - Health check passed
2026-03-31 22:42:09,798 - qm - INFO     - Opening QM
2026-03-31 22:42:09,808 - qm - INFO     - Sending program to QOP for compilation
2026-03-31 22:42:09,968 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 306.00s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 306.08s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 306.15s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 306.23s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 306.31s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 306.39s
Progress: [###################

2026-03-31 22:47:19,282 - qualibrate - INFO - Node dispersive_shift - Execution report for job 1769103657815
No errors


Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 307.37s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 307.46s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 307.51s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 307.57s
2026-03-31 22:47:19,292 - qm - INFO     - Closing QM


2026-03-31 22:47:19,404 - qualibrate - INFO - Node dispersive_shift - Results for qubit q1: SUCCESS
	f_g: 7.50364 GHz (kappa_g FWHM: 1568.1 kHz) | f_e: 7.49669 GHz (kappa_e FWHM: 1361.6 kHz) | chi: -6949.6 kHz | f_opt: 7.50016 GHz
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\20_dispersive_shift.py:188: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-03-31 22:47:19,534 - qualibrate - INFO - Saving node dispersive_shift to local storage
2026-03-31 22:47:19,728 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-03-31 22:47:19,752 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-03-31\#3241_dispersive_shift_224719\quam_state


NodeRunSummary(name='dispersive_shift', description='\n        DISPERSIVE SHIFT (CHI) MEASUREMENT\nThis node measures the dispersive shift chi = f_rr|e - f_rr|g by sweeping the\nreadout resonator frequency in two conditions:\n  1. Qubit in |g⟩ (thermal / after reset)\n  2. Qubit in |e⟩ (after x180 pulse)\n\nThe two Lorentzian dips are fitted; the shift chi and the optimal readout\nfrequency (maximum discrimination contrast) are extracted.\n\nPrerequisites:\n    - Calibrated resonator (nodes 02a/02b).\n    - Calibrated x180 pulse (nodes 04b).\n\nState update:\n    - qubit.resonator.RF_frequency → optimal readout frequency.\n    - qubit.chi (if attribute exists on the qubit object).\n', created_at=datetime.datetime(2026, 3, 31, 22, 42, 6, 569060, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 3, 31, 22, 47, 19, 767145, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Dayligh

### 7b. GEF Dispersive shift (chi_ge, chi_ef)

Sweeps the readout resonator frequency for all three qubit states |g⟩, |e⟩, |f⟩.
Fits a Lorentzian dip to each spectrum and extracts:
- chi_ge = f_resonator(|e⟩) - f_resonator(|g⟩)
- chi_ef = f_resonator(|f⟩) - f_resonator(|e⟩)

Sets the readout frequency to f_resonator(|e⟩) and updates , .

In [6]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

disp_shift_gef = library.nodes["20b_dispersive_shift_gef"].copy(name="dispersive_shift_gef")
disp_shift_gef.parameters.qubits = ["q1"]
disp_shift_gef.parameters.num_shots = 200
disp_shift_gef.parameters.frequency_span_in_mhz = 30.
disp_shift_gef.parameters.frequency_step_in_mhz = 0.05
disp_shift_gef.run()

2026-04-01 00:21:22,454 - qualibrate - INFO - Creating node 20b_dispersive_shift_gef
2026-04-01 00:21:22,514 - qualibrate - INFO - Copying node with name 20b_dispersive_shift_gef with parameters name = 'dispersive_shift_gef', node_parameters = {}
2026-04-01 00:21:22,514 - qualibrate - INFO - Creating node 20b_dispersive_shift_gef
2026-04-01 00:21:22,614 - qualibrate - INFO - Run node dispersive_shift_gef with parameters: {}


2026-04-01 00:21:23,056 - qm - INFO     - Performing health check
2026-04-01 00:21:23,366 - qm - INFO     - Health check passed
2026-04-01 00:21:25,774 - qm - INFO     - Opening QM
2026-04-01 00:21:25,784 - qm - INFO     - Sending program to QOP for compilation
2026-04-01 00:21:26,007 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 606.66s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 606.74s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 606.84s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 606.91s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 606.99s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 607.06s
Progress: [###################

2026-04-01 00:31:39,008 - qualibrate - INFO - Node dispersive_shift_gef - Execution report for job 1769103657824
No errors


Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 609.66s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 609.73s
Progress: [##################################################] 100.0% (n=200/200) --> elapsed time: 609.78s
2026-04-01 00:31:39,017 - qm - INFO     - Closing QM


2026-04-01 00:31:39,128 - qualibrate - INFO - Node dispersive_shift_gef - Results for qubit q1: SUCCESS
	f_g: 7.50369 GHz (kappa_g FWHM: 1727.0 kHz)
	f_e: 7.49677 GHz (kappa_e FWHM: 1532.7 kHz)
	f_f: 7.49186 GHz (kappa_f FWHM: 2095.9 kHz)
	chi_ge: -6927.3 kHz | chi_ef: -4905.5 kHz | f_opt (|e>): 7.49677 GHz
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\20b_dispersive_shift_gef.py:217: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-01 00:31:39,478 - qualibrate - INFO - Saving node dispersive_shift_gef to local storage
2026-04-01 00:31:39,733 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-01 00:31:39,750 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-01\#3249_dispersive_shift_gef_003139\quam_state


NodeRunSummary(name='dispersive_shift_gef', description='\n        GEF DISPERSIVE SHIFT MEASUREMENT\nThis node measures all three resonator frequencies by sweeping the readout\nresonator frequency in three conditions:\n  1. Qubit in |g⟩ (thermal / after reset)\n  2. Qubit in |e⟩ (after x180 pulse)\n  3. Qubit in |f⟩ (after x180 + EF_x180 pulses)\n\nEach spectrum is fitted with a Lorentzian dip. The extracted quantities are:\n  chi_ge = f_resonator(|e⟩) - f_resonator(|g⟩)\n  chi_ef = f_resonator(|f⟩) - f_resonator(|e⟩)\n\nThe optimal readout frequency is set to f_resonator(|e⟩), which gives maximum\ndiscrimination contrast between |g⟩ and |e⟩.\n\nPrerequisites:\n    - Calibrated resonator (nodes 02a/02b).\n    - Calibrated x180 pulse (node 04b).\n    - Calibrated EF_x180 pulse (node 13).\n\nState updates:\n    - qubit.resonator.RF_frequency → f_resonator(|e⟩).\n    - qubit.chi    (if attribute exists) → chi_ge [Hz].\n    - qubit.chi_ef (if attribute exists) → chi_ef [Hz].\n', created_at

### 6h. Selective Power Rabi

Calibrates the amplitude of a narrow-bandwidth `selective_x180` **DragCosinePulse**.
The pulse length controls frequency selectivity: bandwidth ≈ 1/T (e.g. 10 µs → ~100 kHz).

Run the **populate** cell first to write the DragCosinePulse to `state.json`, then run the **rabi** cell to calibrate its amplitude.

**State update**: `operations["selective_x180"].amplitude`, `operations["selective_x180"].length`.

In [ ]:
# Add selective_x180 (DragGaussianPulse) to q1.xy if not already present.
# Parameters are derived from the standard x180 pulse:
#   - length    : set by selective_length_ns below
#   - amplitude : inversely proportional to length  (area = amplitude * length = const)
#   - sigma     : always length / 5
#
# Only inserts missing keys - all other state values are left unchanged.
from quam_config import Quam
from quam.components.pulses import DragGaussianPulse

machine = Quam.load()
xy = machine.qubits["q1"].xy

# Parameters
selective_length_ns = 2000   # adjust as needed

# Read the reference x180 values
x180 = xy.operations["x180"]
x180_length    = x180.length      # ns
x180_amplitude = x180.amplitude   # V

# Scale amplitude inversely with length (constant pulse area -> same rotation angle)
selective_amplitude = x180_amplitude * (x180_length / selective_length_ns)
selective_sigma     = selective_length_ns / 5

print(f"x180 reference : length={x180_length} ns, amplitude={x180_amplitude:.6f} V")
print(f"selective_x180 : length={selective_length_ns} ns, amplitude={selective_amplitude:.6f} V, sigma={selective_sigma:.0f} ns")

if "selective_x180" not in xy.operations:
    xy.operations["selective_x180"] = DragGaussianPulse(
        length=selective_length_ns,
        amplitude=selective_amplitude,
        sigma=selective_sigma,
        alpha=0.0,
        anharmonicity=x180.anharmonicity,
        detuning=0.0,
        subtracted=True,
        axis_angle=0,
        digital_marker="ON",
    )
    print("selective_x180 added")
else:
    print("selective_x180 already exists - skipped (delete it first to re-add)")
    sel = xy.operations["selective_x180"]
    print(f"  current: length={sel.length} ns, amplitude={sel.amplitude:.6f} V, sigma={sel.sigma} ns")

machine.save()
print("State saved.")


In [31]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

selective_rabi = library.nodes["04b_power_rabi"].copy(name="selective_power_rabi")
selective_rabi.parameters.qubits = ["q1"]
selective_rabi.parameters.operation = "selective_x180"
# selective_rabi.parameters.operation_length_in_ns = 4_000  # 10 us -> ~100 kHz bandwidth
selective_rabi.parameters.min_amp_factor = 0.001
selective_rabi.parameters.max_amp_factor = 1.99
selective_rabi.parameters.amp_factor_step = 0.01
selective_rabi.parameters.num_shots = 300
selective_rabi.run()


2026-04-01 14:15:41,783 - qualibrate - INFO - Creating node 04b_power_rabi
2026-04-01 14:15:41,869 - qualibrate - INFO - Copying node with name 04b_power_rabi with parameters name = 'selective_power_rabi', node_parameters = {}
2026-04-01 14:15:41,876 - qualibrate - INFO - Creating node 04b_power_rabi
2026-04-01 14:15:41,969 - qualibrate - INFO - Run node selective_power_rabi with parameters: {}


2026-04-01 14:15:42,252 - qm - INFO     - Performing health check
2026-04-01 14:15:42,906 - qm - INFO     - Health check passed
2026-04-01 14:15:45,484 - qm - INFO     - Opening QM
2026-04-01 14:15:45,494 - qm - INFO     - Sending program to QOP for compilation
2026-04-01 14:15:45,626 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 24.32s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 24.40s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 24.45s


2026-04-01 14:16:10,375 - qualibrate - INFO - Node selective_power_rabi - Execution report for job 1769103657880
No errors


Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 24.50s
2026-04-01 14:16:10,383 - qm - INFO     - Closing QM


2026-04-01 14:16:10,452 - qualibrate - INFO - Node selective_power_rabi - Results for qubit q1:  SUCCESS!
The calibrated selective_x180 amplitude: 9.78 mV (x0.92)
 Rabi periods in sweep: 1.10
 Residual chi2: 0.011
 
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\04b_power_rabi.py:234: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-01 14:16:10,588 - qualibrate - INFO - Saving node selective_power_rabi to local storage
2026-04-01 14:16:10,947 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-01 14:16:10,962 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-01\#3305_selective_power_rabi_141610\quam_state


NodeRunSummary(name='selective_power_rabi', description='\n        POWER RABI WITH ERROR AMPLIFICATION\nThis sequence involves repeatedly executing the qubit pulse (such as x180) \'N\' times and\nmeasuring the state of the resonator across different qubit pulse amplitudes and number of pulses.\nBy doing so, the effect of amplitude inaccuracies is amplified, enabling a more precise measurement of the pi pulse\namplitude. The results are then analyzed to determine the qubit pulse amplitude suitable for the selected duration.\n\nPrerequisites:\n    - Having calibrated the mixer or the Octave (nodes 01a or 01b).\n    - Having calibrated the qubit frequency (node 03a_qubit_spectroscopy.py).\n    - Having set the qubit gates duration (qubit.xy.operations["x180"].length).\n    - Having specified the desired flux point if relevant (qubit.z.flux_point).\n\nState update:\n    - The qubit pulse amplitude corresponding to the specified operation (x180, x90...)\n    (qubit.xy.operations[operation].

## 8. Alice cavity calibration

### Cavity spectroscopy

Sweeps the Alice cavity drive frequency using the `selective_x180` qubit probe to locate the cavity resonance.

**State update**: `cavity_mode.cavity_mode_drive.RF_frequency`

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["21_cavity_mode_spectroscopy"].copy(name="cavity_mode_spectroscopy")
parameters = node.parameters
parameters.num_shots = 200
parameters.mode_name = "alice"
parameters.frequency_span_in_mhz = 2
parameters.frequency_step_in_mhz = 0.1
parameters.operation = "saturation"
parameters.operation_amplitude_factor = 0.1
parameters.operation_len_in_ns = 5000
parameters.cavity_thermalization_time_ns = 1_000_000  # 20 ms = 1x T1; increase if spectrum is noisy
parameters.use_state_discrimination = True
parameters.qubit_probe_operation = "selective_x180"
node.run()

2026-04-01 14:28:38,111 - qualibrate - INFO - Creating node 27_cavity_mode_spectroscopy
2026-04-01 14:28:38,185 - qualibrate - INFO - Copying node with name 27_cavity_mode_spectroscopy with parameters name = 'cavity_mode_spectroscopy', node_parameters = {}
2026-04-01 14:28:38,196 - qualibrate - INFO - Creating node 27_cavity_mode_spectroscopy
2026-04-01 14:28:38,257 - qualibrate - INFO - Run node cavity_mode_spectroscopy with parameters: {}


2026-04-01 14:28:38,557 - qm - INFO     - Performing health check
2026-04-01 14:28:38,867 - qm - INFO     - Health check passed
2026-04-01 14:28:41,410 - qm - INFO     - Opening QM
2026-04-01 14:28:41,420 - qm - INFO     - Sending program to QOP for compilation
2026-04-01 14:28:41,536 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.09s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.15s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.20s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.25s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.29s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 196.34s
Progress: [###################

2026-04-01 14:32:01,850 - qualibrate - INFO - Node cavity_mode_spect... - Execution report for job 1769103657884
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 197.98s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 198.03s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 198.08s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 198.11s
2026-04-01 14:32:01,858 - qm - INFO     - Closing QM


2026-04-01 14:32:01,898 - qualibrate - INFO - Node cavity_mode_spect... - Results for qubit q1: SUCCESS
	Cavity resonance: 5.994736 GHz | FWHM: 2.035 MHz | Detuning offset: 0.470 MHz
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\27_cavity_mode_spectroscopy.py:253: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-01 14:32:01,979 - qualibrate - INFO - Saving node cavity_mode_spectroscopy to local storage
2026-04-01 14:32:02,158 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-01 14:32:02,176 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-01\#3309_cavity_mode_spectroscopy_143202\quam_state


NodeRunSummary(name='cavity_mode_spectroscopy', description="\n        CAVITY MODE SPECTROSCOPY\nFinds the bare resonance frequency of a storage cavity mode (e.g. alice or bob)\nby sweeping the cavity drive frequency and using dispersive coupling to the qubit\nas the photon detector.\n\nSequence (per cavity detuning df):\n  1. Wait 2× thermalization time (qubit and cavity thermalise to |g,0⟩).\n  2. Set qubit drive to bare ge frequency (no sweep on qubit).\n  3. Sweep cavity drive to (IF_cavity + df) and play saturation / probe pulse.\n  4. Apply selective_x180 on qubit at bare ge frequency.\n     - Off resonance (no photons): selective pulse succeeds → qubit in |e⟩.\n     - On resonance (photons present): dispersive shift detunes qubit → pulse\n       fails → qubit stays in |g⟩.\n  5. Measure qubit state.\n\nThe result is a DIP in the qubit excitation probability at the cavity resonance.\nA Lorentzian dip fit extracts the cavity frequency.\n\nPrerequisites:\n    - Calibrated ge and ef

### Displacement calibration

Sweeps the displacement amplitude and measures vacuum-state population using a
selective π-pulse.  Fits P_e(a) = A·exp(−(a/A₁ph)²) to extract the unit
displacement amplitude A₁ph (amplitude_scale=1 → 1 photon).

**State update**: `cavity_mode.cavity_mode_drive.operations["displacement"].amplitude` (Alice)


In [6]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["22_displacement_calibration_vacuum"].copy(name="alice_disp_calib")
parameters = node.parameters
parameters.mode_name = "alice"
parameters.qubit_pulse = "selective_x180"  # or "x180"
parameters.amp_min = 0.0
parameters.amp_max = 1.99
parameters.amp_points = 21
parameters.active_reset = True
parameters.num_shots = 500
parameters.active_reset = True
parameters.cavity_reset_type = "thermal"
parameters.use_state_discrimination = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1]
node.run()


2026-04-06 15:41:06,905 - qualibrate - INFO - Creating node 35_displacement_calibration_vacuum
2026-04-06 15:41:06,985 - qualibrate - INFO - Copying node with name 35_displacement_calibration_vacuum with parameters name = 'alice_disp_calib', node_parameters = {}
2026-04-06 15:41:06,995 - qualibrate - INFO - Creating node 35_displacement_calibration_vacuum
2026-04-06 15:41:07,076 - qualibrate - INFO - Run node alice_disp_calib with parameters: {}


2026-04-06 15:41:07,246 - qm - INFO     - Performing health check
2026-04-06 15:41:07,960 - qm - INFO     - Health check passed
2026-04-06 15:41:10,882 - qm - INFO     - Opening QM
2026-04-06 15:41:10,892 - qm - INFO     - Sending program to QOP for compilation
2026-04-06 15:41:11,061 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 105.05s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 105.10s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 105.14s


2026-04-06 15:42:57,113 - qualibrate - INFO - Node alice_disp_calib - Execution report for job 1769103658127
No errors


Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 105.20s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 105.25s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 105.30s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 105.33s
Progress: [##################################################] 100.0% (n=500/500) --> elapsed time: 105.37s
2026-04-06 15:42:57,123 - qm - INFO     - Closing QM


2026-04-06 15:42:57,153 - qualibrate - INFO - Node alice_disp_calib - [35] q1: SUCCESS | A_1ph (sigma) = 0.9793 | amplitude = 0.363 | offset = 0.140
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\35_displacement_calibration_vacuum.py:272: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-06 15:42:57,273 - qualibrate - INFO - Saving node alice_disp_calib to local storage
2026-04-06 15:42:57,456 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-06 15:42:57,478 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-06\#3510_alice_disp_calib_154257\quam_state


NodeRunSummary(name='alice_disp_calib', description='\n        DISPLACEMENT VACUUM-POPULATION CALIBRATION (35)\n\nCalibrates the unit displacement amplitude by sweeping the cavity displacement\namplitude and measuring the vacuum-state population with a selective qubit π-pulse.\n\nSequence (per displacement amplitude scale a):\n  1. Reset cavity (thermal or active sideband cooling) and qubit.\n  2. Apply displacement pulse at amplitude_scale = a.\n  3. Apply selective_x180 (or x180) on qubit — flips qubit only when cavity is in |0⟩.\n  4. Measure qubit state.\n  5. Apply D(-a) to return cavity toward vacuum (if active_reset = True).\n\nThe measured signal:\n    P_e(a) = amplitude · exp(-(a / A_1ph)²) + offset\n\nwhere A_1ph = sigma is the displacement amplitude_scale that produces exactly 1 photon\non average (n̄ = 1 for a coherent state).\n\nParameters:\n  - mode_name:       Cavity mode to calibrate (\'alice\' or \'bob\').\n  - qubit_pulse:     \'selective_x180\' (spectrally selective,

### Coherent T1

Prepares |α⟩ by displacement, waits variable time t, then probes vacuum population with `selective_x180`. Fits a Gumbel decay to extract T1.

**State update**: `cavity_mode.T1`

In [9]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["23_cavity_coherent_T1"].copy(name="alice_coherent_T1")
parameters = node.parameters
parameters.mode_name = "alice"
parameters.displacement_scale = 1.0   # scale=1 → 1 photon (after node 35); 1.9 → ~3.6 photons
# min/max_wait_time_in_ns are the *per-repeat* range.
# Total sweep spans [min, delay_repeats × max] ns.
parameters.min_wait_time_in_ns = 10_000
parameters.max_wait_time_in_ns = 5_000_000
parameters.wait_time_num_points = 21
parameters.log_or_linear_sweep = "log"      # "log" or "linear"
parameters.delay_repeats = 10
parameters.num_shots = 300
parameters.cavity_reset_type = "thermal"  # "thermal" or "active_sideband"
# parameters.cavity_active_cooling_fock_n = 1
parameters.use_state_discrimination = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1]
node.run()


2026-04-06 16:09:22,461 - qualibrate - INFO - Creating node 33_cavity_coherent_T1
2026-04-06 16:09:22,542 - qualibrate - INFO - Copying node with name 33_cavity_coherent_T1 with parameters name = 'alice_coherent_T1', node_parameters = {}
2026-04-06 16:09:22,552 - qualibrate - INFO - Creating node 33_cavity_coherent_T1


2026-04-06 16:09:22,632 - qualibrate - INFO - Run node alice_coherent_T1 with parameters: {}


2026-04-06 16:09:22,855 - qm - INFO     - Performing health check
2026-04-06 16:09:23,156 - qm - INFO     - Health check passed
2026-04-06 16:09:26,177 - qm - INFO     - Opening QM
2026-04-06 16:09:26,197 - qm - INFO     - Sending program to QOP for compilation
2026-04-06 16:09:36,927 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 118.63s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 118.67s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 118.72s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 118.77s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 118.82s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 118.86s
Progress: [###################

2026-04-06 16:11:36,874 - qualibrate - INFO - Node alice_coherent_T1 - Execution report for job 1769103658130
No errors


Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 118.95s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 119.00s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 119.05s
Progress: [##################################################] 100.0% (n=300/300) --> elapsed time: 119.08s
2026-04-06 16:11:36,875 - qm - INFO     - Closing QM


2026-04-06 16:11:36,972 - qualibrate - INFO - Node alice_coherent_T1 - [33] q1: SUCCESS | T1 = 6628.5 ± 5504.6 µs | nbar0 = 0.16
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\33_cavity_coherent_T1.py:274: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-06 16:11:37,036 - qualibrate - INFO - Saving node alice_coherent_T1 to local storage
2026-04-06 16:11:37,176 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-06 16:11:37,199 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-06\#3513_alice_coherent_T1_161137\quam_state


NodeRunSummary(name='alice_coherent_T1', description="\n        CAVITY COHERENT T1 (33)\n\nMeasures the energy relaxation time T1 of a selected cavity mode by preparing\na coherent state |α⟩ and probing the vacuum-state population with a selective\nqubit π-pulse.\n\nSequence (per wait time t):\n  1. Thermalize cavity (wait ≥ 5×T1) and reset qubit.\n  2. Apply displacement pulse at amplitude_scale = displacement_scale.\n  3. Wait for total time t = delay_repeats × t_per_rep.\n  4. Apply selective_x180 on qubit — flips qubit only when cavity is in |0⟩.\n  5. Measure qubit state.\n\nThe measured signal is:\n    P_e(t) = A · exp(−n̄₀ · exp(−t / T1)) + offset\n\nwhere n̄₀ = displacement_scale² and T1 is the cavity photon lifetime.\n\nParameters:\n  - mode_name:          Cavity mode to probe ('alice' or 'bob').\n  - displacement_scale: Amplitude scale of the displacement pulse.\n                        After node 32 calibration: scale=1 → 1 photon.\n  - t_start_ns / t_end_ns / t_num_points: 

### Photon number resolved spectroscopy

Displaces Alice to |α⟩ and sweeps qubit ge spectroscopy. Photon-number-resolved peaks separated by χ reveal P(n). Use `displacement_alpha` to scale the coherent state amplitude.

**State update**: `cavity_mode.chi`

In [14]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["24_photon_number_resolved_spectroscopy"].copy(name="alice_pns")
parameters = node.parameters
parameters.mode_name = "alice"
parameters.displacement_scale = 1.0
parameters.displacement_alpha = 1.2
parameters.active_reset = False
parameters.qubit_pulse = "selective_x180"  # or "x180"
parameters.right_offset_mhz = 0.5
parameters.left_span_mhz = 1.5
parameters.frequency_step_in_mhz = 0.01
parameters.max_peaks = 3
parameters.chi2_threshold = 2.0
parameters.num_shots = 100
parameters.cavity_reset_type = "active_sideband"
parameters.cavity_active_cooling_fock_n = 4
parameters.f0g1_pulse_duration_ns = 1e6 # 1 ms
parameters.use_state_discrimination = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1]
node.run()


2026-04-08 00:10:17,051 - qualibrate - INFO - Creating node 24_photon_number_splitting


2026-04-08 00:10:17,141 - qualibrate - INFO - Copying node with name 24_photon_number_splitting with parameters name = 'alice_pns', node_parameters = {}
2026-04-08 00:10:17,151 - qualibrate - INFO - Creating node 24_photon_number_splitting
2026-04-08 00:10:17,232 - qualibrate - INFO - Run node alice_pns with parameters: {}
c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=1000000.0, input_type=float])
  return self.__pydantic_serializer__.to_python(


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-04-08 00:10:17,552 - qm - INFO     - Performing health check
2026-04-08 00:10:17,854 - qm - INFO     - Health check passed
2026-04-08 00:10:20,626 - qm - INFO     - Opening QM
2026-04-08 00:10:20,636 - qm - INFO     - Sending program to QOP for compilation
2026-04-08 00:10:21,047 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 123.73s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 123.78s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 123.83s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 123.88s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 123.93s
Progress: [#######################

2026-04-08 00:12:27,393 - qualibrate - INFO - Node alice_pns - Execution report for job 1769103658386
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 125.01s
2026-04-08 00:12:27,393 - qm - INFO     - Closing QM


2026-04-08 00:12:27,433 - qualibrate - INFO - Node alice_pns - [29] q1: FAIL | chi = nan kHz | peaks = 1 | positions (kHz) = ['-726.424'] | P(n) = ['1.000']
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\24_photon_number_splitting.py:278: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-08 00:12:27,533 - qualibrate - INFO - Saving node alice_pns to local storage


Action execute_qua_program finished
Running action analyse_data
Action analyse_data finished
Running action plot_data
Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-04-08 00:12:27,720 - qualibrate - INFO - Saving machine state to db
2026-04-08 00:12:27,734 - qualibrate - WARNING - save failed: No database connection configured for project 'calib_1q'
2026-04-08 00:12:27,734 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-08 00:12:27,759 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-08\#3559_alice_pns_001227\quam_state


Action save_results finished


NodeRunSummary(name='alice_pns', description="\n        PHOTON NUMBER SPLITTING — CHI MEASUREMENT (29)\n\nDisplaces the selected cavity mode to a coherent state |α⟩ and sweeps the\nqubit ge spectroscopy frequency.  The resulting spectrum shows photon-number-\nsplit peaks:\n\n    f_n = f_q - 2*chi*n   (n=0, 1, 2, ...)\n\nseparated by 2*chi.  The node auto-detects the number of peaks (1 → max_peaks)\nby fitting successive multi-Gaussian models until the reduced chi² drops below\nthe threshold.  The mean spacing between adjacent peaks = 2*chi is reported and\nsaved to the machine state.\n\nAfter measurement an optional active reset applies D(-α) to return the cavity\nto vacuum immediately, replacing passive thermalization.\n\nPrerequisites:\n    - Calibrated qubit_pulse operation on qubit.xy (e.g. selective_x180 or x180).\n    - A 'displacement' operation on cavity_mode_drive.\n\nParameters:\n    - displacement_scale:  amplitude scale for the displacement pulse.\n                         

### Dispersive shift χ (Ramsey Stark)

Measures χ between the qubit and Alice cavity by Ramsey interferometry with a CW cavity drive.
The qubit frequency shifts by Δf = 2χ n̅ as the cavity photon number increases.

**State update**: `cavity_mode.chi` and `cavity_transmon_pairs["q1_alice"].chi`

In [9]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["25_chi_ramsey_stark"].copy(name="alice_chi_ramsey")
parameters = node.parameters
parameters.mode_name = "alice"
parameters.num_shots = 100
parameters.min_delay_ns = 16
parameters.max_delay_ns = 15_000
parameters.delay_step_ns = 128
parameters.ring_up_ns = 2000
parameters.cavity_amplitudes = [0.0, 1.0, 1.99]
parameters.artificial_detuning_hz = 0.2e6
parameters.use_state_discrimination = True
parameters.cavity_reset_type = "thermal"
node.run()

2026-04-07 23:36:41,075 - qualibrate - INFO - Creating node 25_chi_ramsey_stark
2026-04-07 23:36:41,431 - qualibrate - INFO - Copying node with name 25_chi_ramsey_stark with parameters name = 'alice_chi_ramsey', node_parameters = {}
2026-04-07 23:36:41,431 - qualibrate - INFO - Creating node 25_chi_ramsey_stark
2026-04-07 23:36:41,511 - qualibrate - INFO - Run node alice_chi_ramsey with parameters: {}
c:\Users\td-srv-quantum\.conda\envs\qualibrate\Lib\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `int` - serialized value may not be as expected [input_value=200000.0, input_type=float])
  return self.__pydantic_serializer__.to_python(


Running action create_qua_program
Action create_qua_program finished
Running action execute_qua_program
2026-04-07 23:36:41,841 - qm - INFO     - Performing health check
2026-04-07 23:36:42,281 - qm - INFO     - Health check passed
2026-04-07 23:36:45,675 - qm - INFO     - Opening QM
2026-04-07 23:36:45,695 - qm - INFO     - Sending program to QOP for compilation
2026-04-07 23:36:45,866 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 234.07s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 234.11s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 234.16s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 234.20s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 234.24s
Progress: [#######################

2026-04-07 23:40:45,073 - qualibrate - INFO - Node alice_chi_ramsey - Execution report for job 1769103658381
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 236.63s
2026-04-07 23:40:45,073 - qm - INFO     - Closing QM


2026-04-07 23:40:45,143 - qualibrate - INFO - Node alice_chi_ramsey - q1: Δf/A² slope = 0.035 MHz/A²  (fit RMS = 18.2 kHz)
2026-04-07 23:40:45,143 - qualibrate - INFO - Node alice_chi_ramsey - q1: χ = 0.017 MHz  (via displacement_k)


...

C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\25_chi_ramsey_stark.py:357: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-07 23:40:45,304 - qualibrate - INFO - Saving node alice_chi_ramsey to local storage


Action plot_data finished
Running action update_state
Action update_state finished
Running action save_results


2026-04-07 23:40:45,550 - qualibrate - INFO - Saving machine state to db
2026-04-07 23:40:45,567 - qualibrate - WARNING - save failed: No database connection configured for project 'calib_1q'
2026-04-07 23:40:45,567 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-07 23:40:45,596 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-07\#3554_alice_chi_ramsey_234045\quam_state


Action save_results finished


NodeRunSummary(name='alice_chi_ramsey', description='\n        RAMSEY STARK-SHIFT — CAVITY-TRANSMON χ CALIBRATION (25)\n\nMeasures the dispersive shift χ between a transmon qubit and a storage cavity\nmode using Ramsey interferometry under a continuous-wave (CW) cavity drive.\n\nPhysics\n-------\nIn the dispersive regime:\n\n    H/ħ = ω_r a†a  +  (ω_q/2) σ_z  +  χ a†a σ_z\n\nThe qubit frequency shifts by\n\n    Δω_q = 2χ n̄_ss\n\nwhen the cavity is populated with a steady-state photon number n̄_ss ∝ A²,\nwhere A is the (dimensionless) cavity drive amplitude.\n\nExperiment sequence\n-------------------\nFor each (A, τ) pair:\n\n  1. Reset cavity (thermal or active sideband) and qubit.\n  2. Apply CW cavity drive at amplitude A for a fixed duration\n       T_total = ring_up_ns + max_delay_ns + 2 × t_x90 + buffer\n     The cavity reaches steady state n̄_ss ∝ A² after ring_up_ns.\n  3. Wait ring_up_ns on the qubit channel (cavity still being driven).\n  4. Ramsey with artificial detuning:\

### f0g1 sideband calibration

#### Spectroscopy

Sweeps f0g1 drive frequency while qubit is in |f⟩. Resonance shows as a dip in qubit state population.

**State update**: `sideband_drive.RF_frequency`

In [12]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

alice_spec = library.nodes["26_f0g1_spectroscopy"].copy(name="alice_f0g1_spec")
alice_spec.parameters.qubits = ["q1"]
alice_spec.parameters.mode_name = "alice"
alice_spec.parameters.operation = "f0g1_pi"
alice_spec.parameters.frequency_span_in_mhz = 10
alice_spec.parameters.frequency_step_in_mhz = 0.05
alice_spec.parameters.operation_len_in_ns = 1_000
alice_spec.parameters.operation_amplitude_factor = 1.0
alice_spec.parameters.num_shots = 100
alice_spec.parameters.cavity_thermalization_time_ns = 1_000_000  # 20 ms = 1x T1
alice_spec.run()

2026-04-06 11:36:57,001 - qualibrate - INFO - Creating node 21_f0g1_spectroscopy
2026-04-06 11:36:57,066 - qualibrate - INFO - Copying node with name 21_f0g1_spectroscopy with parameters name = 'alice_f0g1_spec', node_parameters = {}
2026-04-06 11:36:57,066 - qualibrate - INFO - Creating node 21_f0g1_spectroscopy
2026-04-06 11:36:57,137 - qualibrate - INFO - Run node alice_f0g1_spec with parameters: {}


2026-04-06 11:36:57,398 - qm - INFO     - Performing health check
2026-04-06 11:36:57,698 - qm - INFO     - Health check passed
2026-04-06 11:37:00,317 - qm - INFO     - Opening QM
2026-04-06 11:37:00,337 - qm - INFO     - Sending program to QOP for compilation
2026-04-06 11:37:00,568 - qm - INFO     - Executing program
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 114.93s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 114.99s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 115.03s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 115.08s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 115.12s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 115.17s
Progress: [###################

2026-04-06 11:38:58,082 - qualibrate - INFO - Node alice_f0g1_spec - Execution report for job 1769103658108
No errors


Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 116.04s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 116.08s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 116.13s
Progress: [##################################################] 100.0% (n=100/100) --> elapsed time: 116.16s
2026-04-06 11:38:58,082 - qm - INFO     - Closing QM


2026-04-06 11:38:58,162 - qualibrate - INFO - Node alice_f0g1_spec - Results for qubit q1: SUCCESS
	f0g1 frequency: 3.3175 GHz | FWHM: 2031.0 kHz
C:\dev_packages\qualibrate\qua-libs\qualibration_graphs\superconducting\calibrations\1Q_calibrations\21_f0g1_spectroscopy.py:254: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
2026-04-06 11:38:58,242 - qualibrate - INFO - Saving node alice_f0g1_spec to local storage
2026-04-06 11:38:58,350 - qualibrate - INFO - Saving machine to active path D:\MData\DR3-Run009\qm_calib\quam_state
2026-04-06 11:38:58,377 - qualibrate - INFO - Saving machine to data folder D:\MData\DR3-Run009\qm_calib\storage\2026-04-06\#3492_alice_f0g1_spec_113858\quam_state


NodeRunSummary(name='alice_f0g1_spec', description='\n        F0G1 SPECTROSCOPY\nSweeps the f0g1 sideband drive frequency while the qubit is prepared in |f⟩.\nWhen the sideband drive is resonant, the |f,0⟩ ↔ |g,1⟩ transition is driven;\nthe qubit is left in |g⟩ and the back-swap π_ef leaves it in |g⟩ → DIP in\nstate measurement.\n\nSequence:\n  1. Wait thermalization time (2× T1)\n  2. π_ge  →  |e⟩\n  3. π_ef  →  |f⟩\n  4. Sweep f0g1 IF;  play saturation pulse on f0g1 channel\n  5. π_ef  (back-swap: |f⟩ → |e⟩ if no photon created; |g⟩ unchanged)\n  6. Measure qubit state\n\nPrerequisites:\n    - Calibrated ge and ef transitions (nodes 04b, 13).\n\nState update:\n    - cavity_transmon_pairs["{qubit}_{mode}"].sideband_drive.RF_frequency  →  sideband resonance frequency.\n', created_at=datetime.datetime(2026, 4, 6, 11, 36, 57, 147940, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=68400), 'Central Daylight Time')), completed_at=datetime.datetime(2026, 4, 6, 11, 38, 58, 40820

#### Time Rabi

Sweeps f0g1 drive duration to calibrate the π-pulse length.

**State update**: `sideband_drive.operations["f0g1_pi"].length`

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

alice_time_rabi = library.nodes["27_f0g1_time_rabi"].copy(name="alice_f0g1_time_rabi")
alice_time_rabi.parameters.qubits = ["q1"]
alice_time_rabi.parameters.mode_name = "alice"
alice_time_rabi.parameters.min_duration_ns = 16
alice_time_rabi.parameters.max_duration_ns = 2000
alice_time_rabi.parameters.duration_step_ns = 4
alice_time_rabi.parameters.cavity_thermalization_time_ns = 20_000_000  # 20 ms = 1x T1
alice_time_rabi.run()

### f0g1 cavity T1

Encodes one photon via f0g1 π-pulse, waits τ, retrieves and measures. Population vs τ gives T1_Alice.

**State update**: `cavity_mode.T1`

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

alice_T1 = library.nodes["28_cavity_mode_T1"].copy(name="alice_cavity_mode_T1")
alice_T1.parameters.qubits = ["q1"]
alice_T1.parameters.mode_name = "alice"
# Set max idle time to cover the expected cavity T1 range (e.g. up to 500 µs):
# alice_T1.parameters.max_wait_time_in_ns = 500_000
alice_T1.run()

### Coherent T2 Ramsey

Ramsey on the Fock-state superposition |0⟩+|1⟩ to measure T2* of Alice. Requires calibrated f0g1 π-pulse.

**State update**: `cavity_mode.T2ramsey`

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

alice_T2 = library.nodes["29_cavity_mode_T2"].copy(name="alice_cavity_mode_T2")
alice_T2.parameters.qubits = ["q1"]
alice_T2.parameters.mode_name = "alice"
alice_T2.parameters.ramsey_detuning_hz = 1000.0
# alice_T2.parameters.max_wait_time_in_ns = 100_000
alice_T2.run()

## 9. Bob cavity calibration

### Cavity spectroscopy

**State update**: `cavity_mode.cavity_mode_drive.RF_frequency` (Bob)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["21_cavity_mode_spectroscopy"].copy(name="cavity_mode_spectroscopy_bob")
node.parameters.mode_name = "bob"
node.parameters.frequency_span_in_mhz = 400.0
node.parameters.frequency_step_in_mhz = 1
node.parameters.cavity_thermalization_time_ns = 20_000_000  # 20 ms = 1x T1; increase if spectrum is noisy
node.run()

### Displacement calibration

Sweeps the displacement amplitude and measures vacuum-state population using a
selective π-pulse.  Fits P_e(a) = A·exp(−(a/A₁ph)²) to extract the unit
displacement amplitude A₁ph (amplitude_scale=1 → 1 photon).

**State update**: `cavity_mode.cavity_mode_drive.operations["displacement"].amplitude` (Bob)


In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["22_displacement_calibration_vacuum"].copy(name="bob_disp_calib")
parameters = node.parameters
parameters.mode_name = "bob"
parameters.qubit_pulse = "selective_x180"  # or "x180"
parameters.amp_min = 0.0
parameters.amp_max = 2.0
parameters.amp_points = 51
parameters.active_reset = True
parameters.num_shots = 1000
parameters.use_state_discrimination = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1]
node.run()


### Coherent T1

**State update**: `cavity_mode.T1` (Bob)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["23_cavity_coherent_T1"].copy(name="bob_coherent_T1")
parameters = node.parameters
parameters.mode_name = "bob"
parameters.displacement_scale = 1.9
parameters.min_wait_time_in_ns = 16
parameters.max_wait_time_in_ns = 5_000_000
parameters.wait_time_num_points = 51
parameters.log_or_linear_sweep = "log"      # "log" or "linear"
parameters.delay_repeats = 1
parameters.num_shots = 1000
parameters.cavity_reset_type = "thermal"  # "thermal" or "active_sideband"
parameters.cavity_active_cooling_fock_n = 1
parameters.use_state_discrimination = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1]
node.run()


### Photon number resolved spectroscopy

Use `displacement_alpha` to scale the coherent state amplitude.

**State update**: `cavity_mode.chi` (Bob)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["24_photon_number_resolved_spectroscopy"].copy(name="bob_pns")
parameters = node.parameters
parameters.mode_name = "bob"
parameters.displacement_scale = 1.5
parameters.displacement_alpha = 1.0
parameters.active_reset = True
parameters.qubit_pulse = "selective_x180"  # or "x180"
parameters.right_offset_mhz = 2.0
parameters.left_span_mhz = 6.0
parameters.frequency_step_in_mhz = 0.04
parameters.max_peaks = 8
parameters.chi2_threshold = 2.0
parameters.num_shots = 100
parameters.use_state_discrimination = True
parameters.normalize_plot = False  # set True when use_state_discrimination=False to normalize I to [0,1]
node.run()


### Dispersive shift χ (Ramsey Stark)

Measures χ between the qubit and Bob cavity by Ramsey interferometry with a CW cavity drive.

**State update**: `cavity_mode.chi` and `cavity_transmon_pairs["q1_bob"].chi`

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()
node = library.nodes["25_chi_ramsey_stark"].copy(name="bob_chi_ramsey")
parameters = node.parameters
parameters.mode_name = "bob"
parameters.num_shots = 1000
parameters.min_delay_ns = 16
parameters.max_delay_ns = 3000
parameters.delay_step_ns = 16
parameters.ring_up_ns = 2000
parameters.cavity_amplitudes = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25]
parameters.artificial_detuning_hz = 200_000
parameters.use_state_discrimination = True
parameters.cavity_reset_type = "thermal"
node.run()

### f0g1 sideband calibration

#### Spectroscopy

**State update**: `sideband_drive.RF_frequency` (Bob)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

bob_spec = library.nodes["26_f0g1_spectroscopy"].copy(name="bob_f0g1_spec")
bob_spec.parameters.qubits = ["q1"]
bob_spec.parameters.mode_name = "bob"
bob_spec.parameters.frequency_span_in_mhz = 100
bob_spec.parameters.frequency_step_in_mhz = 0.25
bob_spec.parameters.cavity_thermalization_time_ns = 20_000_000  # 20 ms = 1x T1
bob_spec.run()

#### Time Rabi

**State update**: `sideband_drive.operations["f0g1_pi"].length` (Bob)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

bob_time_rabi = library.nodes["27_f0g1_time_rabi"].copy(name="bob_f0g1_time_rabi")
bob_time_rabi.parameters.qubits = ["q1"]
bob_time_rabi.parameters.mode_name = "bob"
bob_time_rabi.parameters.min_duration_ns = 16
bob_time_rabi.parameters.max_duration_ns = 2000
bob_time_rabi.parameters.duration_step_ns = 4
bob_time_rabi.parameters.cavity_thermalization_time_ns = 20_000_000  # 20 ms = 1x T1
bob_time_rabi.run()

### f0g1 cavity T1

**State update**: `cavity_mode.T1` (Bob, f0g1 method)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

bob_T1 = library.nodes["28_cavity_mode_T1"].copy(name="bob_cavity_mode_T1")
bob_T1.parameters.qubits = ["q1"]
bob_T1.parameters.mode_name = "bob"
# bob_T1.parameters.max_wait_time_in_ns = 500_000
bob_T1.run()

### Coherent T2 Ramsey

Requires calibrated f0g1 π-pulse.

**State update**: `cavity_mode.T2ramsey` (Bob)

In [ ]:
from qualibrate import QualibrationLibrary
library = QualibrationLibrary.get_active_library()

bob_T2 = library.nodes["29_cavity_mode_T2"].copy(name="bob_cavity_mode_T2")
bob_T2.parameters.qubits = ["q1"]
bob_T2.parameters.mode_name = "bob"
bob_T2.parameters.ramsey_detuning_hz = 1000.0
# bob_T2.parameters.max_wait_time_in_ns = 100_000
bob_T2.run()

## 10. Automated calibration graphs

### 10a. SRF bring-up graph

Full automated bring-up:
```
resonator_spec → qubit_spec → power_rabi → iq_blobs → readout_opt → T1_ge
    → qubit_spec_ef → power_rabi_ef → dispersive_shift
        → alice_spec → alice_rabi → alice_T1
        → bob_spec   → bob_rabi   → bob_T1
```

In [ ]:
from typing import List
from qualibrate.orchestration.basic_orchestrator import BasicOrchestrator
from qualibrate.parameters import GraphParameters
from qualibrate.qualibration_graph import QualibrationGraph
from qualibrate.qualibration_library import QualibrationLibrary

library = QualibrationLibrary.get_active_library()


class Parameters(GraphParameters):
    qubits: List[str] = ["q1"]


g_srf_bringup = QualibrationGraph(
    name="SRF_BringUp",
    parameters=Parameters(),
    nodes={
        # --- Readout resonator ---
        "resonator_spec": library.nodes["02a_resonator_spectroscopy"].copy(
            name="resonator_spec"),
        # --- Transmon ge ---
        "qubit_spec": library.nodes["03a_qubit_spectroscopy"].copy(
            name="qubit_spec",
            find_dip=True,
            frequency_span_in_mhz=100,
        ),
        "power_rabi": library.nodes["04b_power_rabi"].copy(name="power_rabi"),
        "iq_blobs": library.nodes["07_iq_blobs"].copy(name="iq_blobs"),
        "readout_opt": library.nodes["08a_readout_frequency_optimization"].copy(
            name="readout_opt"),
        "T1_ge": library.nodes["05_T1"].copy(
            name="T1_ge", use_state_discrimination=True),
        # --- Transmon ef ---
        "qubit_spec_ef": library.nodes["12_qubit_spectroscopy_EF"].copy(
            name="qubit_spec_ef", find_dip=True),
        "power_rabi_ef": library.nodes["13_power_rabi_ef"].copy(
            name="power_rabi_ef"),
        # --- Transmon-cavity coupling ---
        "dispersive_shift": library.nodes["20_dispersive_shift"].copy(
            name="dispersive_shift"),
        # --- Alice cavity mode ---
        "alice_spec": library.nodes["26_f0g1_spectroscopy"].copy(
            name="alice_spec", mode_name="alice"),
        "alice_time_rabi": library.nodes["27_f0g1_time_rabi"].copy(
            name="alice_time_rabi", mode_name="alice"),
        "alice_T1": library.nodes["28_cavity_mode_T1"].copy(
            name="alice_T1", mode_name="alice"),
        # --- Bob cavity mode ---
        "bob_spec": library.nodes["26_f0g1_spectroscopy"].copy(
            name="bob_spec", mode_name="bob"),
        "bob_time_rabi": library.nodes["27_f0g1_time_rabi"].copy(
            name="bob_time_rabi", mode_name="bob"),
        "bob_T1": library.nodes["28_cavity_mode_T1"].copy(
            name="bob_T1", mode_name="bob"),
    },
    connectivity=[
        ("resonator_spec",   "qubit_spec"),
        ("qubit_spec",       "power_rabi"),
        ("power_rabi",       "iq_blobs"),
        ("iq_blobs",         "readout_opt"),
        ("readout_opt",      "T1_ge"),
        ("T1_ge",            "qubit_spec_ef"),
        ("qubit_spec_ef",    "power_rabi_ef"),
        ("power_rabi_ef",    "dispersive_shift"),
        ("dispersive_shift", "alice_spec"),
        ("alice_spec",       "alice_time_rabi"),
        ("alice_time_rabi",  "alice_T1"),
        ("dispersive_shift", "bob_spec"),
        ("bob_spec",         "bob_time_rabi"),
        ("bob_time_rabi",    "bob_T1"),
    ],
    orchestrator=BasicOrchestrator(skip_failed=False),
)

g_srf_bringup.run()

### 10b. Cavity maintenance graph

Quick re-tuning: re-check qubit ge frequency and both cavity sideband frequencies.

In [ ]:
from typing import List
from qualibrate.orchestration.basic_orchestrator import BasicOrchestrator
from qualibrate.parameters import GraphParameters
from qualibrate.qualibration_graph import QualibrationGraph
from qualibrate.qualibration_library import QualibrationLibrary

library = QualibrationLibrary.get_active_library()


class Parameters(GraphParameters):
    qubits: List[str] = ["q1"]


g_maintenance = QualibrationGraph(
    name="SRF_Maintenance",
    parameters=Parameters(),
    nodes={
        "qubit_spec": library.nodes["03a_qubit_spectroscopy"].copy(
            name="qubit_spec",
            find_dip=True,
            frequency_span_in_mhz=20,
        ),
        "alice_spec": library.nodes["26_f0g1_spectroscopy"].copy(
            name="alice_spec", mode_name="alice", frequency_span_in_mhz=20),
        "bob_spec": library.nodes["26_f0g1_spectroscopy"].copy(
            name="bob_spec", mode_name="bob", frequency_span_in_mhz=20),
    },
    connectivity=[
        ("qubit_spec", "alice_spec"),
        ("qubit_spec", "bob_spec"),
    ],
    orchestrator=BasicOrchestrator(skip_failed=True),
)

g_maintenance.run()